In [ ]:
import os

print("Рабочая папка:", os.getcwd())


In [ ]:

!pip install -q kagglehub


In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "preethi866/vegetable-dataset"
)

print("Путь к датасету:", dataset_path)


In [ ]:

import os

print(os.listdir(dataset_path))


In [ ]:

dataset_folder = os.path.join(dataset_path, "Dataset")
final_folder = os.path.join(dataset_path, "final_dataset")

print("Содержимое Dataset:")
print(os.listdir(dataset_folder))

print("\nСодержимое final_dataset:")
print(os.listdir(final_folder))


In [ ]:

dataset_inner = os.path.join(dataset_folder, "Dataset")
final_inner = os.path.join(final_folder, "final_dataset")

print("Внутри Dataset/Dataset:")
print(os.listdir(dataset_inner))

print("\nВнутри final_dataset/final_dataset:")
print(os.listdir(final_inner))


In [ ]:

from pathlib import Path
from PIL import Image
import pandas as pd
import hashlib

# 1. Указываем, где находятся данные

root = Path(dataset_path)

folders = {
    "Dataset": root / "Dataset" / "Dataset",
    "train": root / "final_dataset" / "final_dataset" / "train",
    "val": root / "final_dataset" / "final_dataset" / "val",
    "test": root / "final_dataset" / "final_dataset" / "test"
}

classes = [
    "Capsicum", "Carrot", "Cucumber",
    "Potato", "Raddish", "Squash"
]

extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

# 2. Функция вычисления SHA-256

def calculate_hash(file_path):
    sha = hashlib.sha256()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            sha.update(chunk)

    return sha.hexdigest()


# 3. Собираем информацию обо всех фотографиях

records = []
problems = []

for section, base_folder in folders.items():
    for vegetable in classes:

        class_folder = base_folder / vegetable

        if not class_folder.is_dir():
            problems.append(f"Отсутствует папка: {class_folder}")
            continue

        for file_path in class_folder.rglob("*"):

            if not file_path.is_file():
                continue

            if file_path.suffix.lower() not in extensions:
                problems.append(f"Неизвестный формат: {file_path}")
                continue

            try:
                with Image.open(file_path) as image:
                    width, height = image.size
                    image.verify()

                file_hash = calculate_hash(file_path)

                records.append({
                    "section": section,
                    "class": vegetable,
                    "filename": file_path.name,
                    "extension": file_path.suffix.lower(),
                    "width": width,
                    "height": height,
                    "hash": file_hash
                })

            except Exception as error:
                problems.append(f"{file_path}: {error}")

# Превращаем результаты в таблицу
df = pd.DataFrame(records)

print("Всего успешно проверено изображений:", len(df))


# 4. Количество изображений по разделам и классам

counts = pd.crosstab(df["section"], df["class"])

counts = counts.reindex(
    index=["Dataset", "train", "val", "test"],
    columns=classes,
    fill_value=0
)

counts["TOTAL"] = counts.sum(axis=1)

print("\n1. КОЛИЧЕСТВО ИЗОБРАЖЕНИЙ")
display(counts)


# 5. Форматы файлов

print("\n2. ФОРМАТЫ ИЗОБРАЖЕНИЙ")

formats = pd.crosstab(df["section"], df["extension"])
display(formats)


# 6. Самые распространённые размеры изображений

print("\n3. ТОП-10 РАЗМЕРОВ ИЗОБРАЖЕНИЙ")

sizes = (
    df.groupby(["width", "height"])
    .size()
    .sort_values(ascending=False)
    .head(10)
)

display(sizes)


# 7. Примеры названий файлов

print("\n4. ПРИМЕРЫ НАЗВАНИЙ ФАЙЛОВ")

for vegetable in classes:
    examples = df[
        (df["section"] == "Dataset") &
        (df["class"] == vegetable)
    ]["filename"].head(3).tolist()

    print(f"{vegetable}: {examples}")


# 8. Проверяем точные дубликаты в Dataset

print("\n5. ДУБЛИКАТЫ В ИСХОДНОМ DATASET")

original = df[df["section"] == "Dataset"]

duplicate_groups = original.groupby("hash").size()
duplicate_groups = duplicate_groups[duplicate_groups > 1]

print("Групп одинаковых файлов:", len(duplicate_groups))
print("Лишних повторных копий:", int((duplicate_groups - 1).sum()))


# 9. Проверяем утечку между train, val и test

print("\n6. ДУБЛИКАТЫ МЕЖДУ TRAIN / VAL / TEST")

split_data = df[df["section"].isin(["train", "val", "test"])]

hash_splits = split_data.groupby("hash")["section"].nunique()
leaked_hashes = hash_splits[hash_splits > 1].index

print("Групп файлов, встречающихся в разных разделах:", len(leaked_hashes))

if len(leaked_hashes) > 0:
    print("\nПримеры совпадений:")

    examples = split_data[
        split_data["hash"].isin(leaked_hashes)
    ][["section", "class", "filename", "hash"]]

    display(examples.sort_values("hash").head(15))


# 10. Проверяем ошибки

print("\n7. ПРОБЛЕМНЫЕ ФАЙЛЫ")
print("Количество проблем:", len(problems))

for problem in problems[:10]:
    print(problem)


In [ ]:

import re

# Берём только исходный Dataset
original = df[df["section"] == "Dataset"].copy()

# Определяем предполагаемую съёмку по имени файла
def get_shooting(filename):
    name = Path(filename).stem
    return re.sub(r"_frame_\d+$", "", name)

original["shooting"] = original["filename"].apply(get_shooting)

# Считаем фотографии внутри каждой съёмки
groups = (
    original.groupby(["class", "shooting"])
    .size()
    .reset_index(name="photos")
)

# Общая статистика по классам
summary = groups.groupby("class").agg(
    shootings=("shooting", "count"),
    largest_group=("photos", "max")
)

summary["total_photos"] = original.groupby("class").size()

summary["largest_group_percent"] = (
    summary["largest_group"] / summary["total_photos"] * 100
).round(1)

print("1. КОЛИЧЕСТВО ПРЕДПОЛАГАЕМЫХ СЪЁМОК")
display(summary)

print("\n2. САМЫЕ КРУПНЫЕ ГРУППЫ")
display(
    groups.sort_values("photos", ascending=False).head(15)
)


In [ ]:

result = []

for vegetable in classes:
    class_groups = groups[groups["class"] == vegetable]
    sizes = class_groups["photos"]

    result.append({
        "class": vegetable,
        "1 photo": (sizes == 1).sum(),
        "2-9 photos": sizes.between(2, 9).sum(),
        "10+ photos": (sizes >= 10).sum(),
        "photos in large groups": sizes[sizes >= 10].sum()
    })

group_analysis = pd.DataFrame(result)

display(group_analysis)

In [ ]:

import matplotlib.pyplot as plt
from PIL import Image

# Выбираем четыре самые большие группы
largest = groups.sort_values(
    "photos", ascending=False
).head(4)

fig, axes = plt.subplots(4, 5, figsize=(15, 10))

for row_number, (_, group) in enumerate(largest.iterrows()):

    vegetable = group["class"]
    shooting = group["shooting"]

    # Находим фотографии нужной съёмки
    frames = original[
        (original["class"] == vegetable) &
        (original["shooting"] == shooting)
    ].copy()

    # Сортируем по номеру кадра
    frames["frame_number"] = frames["filename"].str.extract(
        r"_frame_(\d+)"
    )[0].astype(int)

    frames = frames.sort_values("frame_number")

    # Берём пять кадров из разных частей серии
    positions = [
        round(i * (len(frames) - 1) / 4)
        for i in range(5)
    ]

    for column, position in enumerate(positions):
        filename = frames.iloc[position]["filename"]
        image_path = Path(dataset_inner) / vegetable / filename

        with Image.open(image_path) as image:
            axes[row_number, column].imshow(image.convert("RGB"))

        axes[row_number, column].set_title(
            f"{vegetable}: {frames.iloc[position]['frame_number']}",
            fontsize=10
        )
        axes[row_number, column].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

# Получаем хеши исходного Dataset
source_hashes = set(
    df[df["section"] == "Dataset"]["hash"]
)

# Получаем хеши из train, val и test
final_hashes = set(
    df[df["section"] != "Dataset"]["hash"]
)

print("Уникальных изображений в Dataset:", len(source_hashes))
print("Уникальных изображений в final_dataset:", len(final_hashes))

print("\nСовпадают между наборами:",
      len(source_hashes & final_hashes))

print("Только в Dataset:",
      len(source_hashes - final_hashes))

print("Только в final_dataset:",
      len(final_hashes - source_hashes))


In [ ]:

splits = {}

for name in ["train", "val", "test"]:
    splits[name] = set(
        df[df["section"] == name]["hash"]
    )

print("Одинаковые изображения между разделами:")
print("Train и Val:", len(splits["train"] & splits["val"]))
print("Train и Test:", len(splits["train"] & splits["test"]))
print("Val и Test:", len(splits["val"] & splits["test"]))

print("\nПрисутствуют во всех трёх:",
      len(splits["train"] & splits["val"] & splits["test"]))

labels = df.groupby("hash")["class"].nunique()

print("\nФотографий с противоречивыми классами:",
      (labels > 1).sum())


In [ ]:

# Берём информацию об исходных фотографиях
audit = original.copy()

# Извлекаем дату из названия предполагаемой съёмки
audit["date"] = audit["shooting"].str[:8]

# Определяем, является ли файл кадром видеозаписи
audit["is_frame"] = audit["filename"].str.contains(
    r"_frame_\d+\.jpg$", regex=True
)

# Группируем фотографии по классу и дате
date_summary = audit.groupby(["class", "date"]).agg(
    total=("filename", "size"),
    video_frames=("is_frame", "sum"),
    groups=("shooting", "nunique")
).reset_index()

# Подсчитываем отдельные фотографии
date_summary["single_photos"] = (
    date_summary["total"] - date_summary["video_frames"]
)

display(date_summary)


In [ ]:

# Берём только крупные серии кадров
series = groups[groups["photos"] >= 10].copy()

# Преобразуем дату и время из названия в настоящий формат времени
series["time"] = pd.to_datetime(
    series["shooting"],
    format="%Y%m%d_%H%M%S"
)

# Выделяем календарную дату
series["date"] = series["time"].dt.date

# Сортируем серии по классу и времени
series = series.sort_values(["class", "time"])

# Считаем разницу в минутах между началами соседних серий
series["gap_minutes"] = (
    series.groupby(["class", "date"])["time"]
    .diff()
    .dt.total_seconds() / 60
).round(1)

display(series[[
    "class", "shooting", "photos", "gap_minutes"
]])

#Nan - здесь означает, что программе не с чем сравнить время этой серии: она первая для данного овоща в этот день.


In [ ]:

# Выбираем фотографии без номера frame
singles = original[
    ~original["filename"].str.contains(
        r"_frame_\d+\.jpg$", regex=True
    )
].copy()

# Получаем время одиночной фотографии
singles["photo_time"] = pd.to_datetime(
    singles["shooting"],
    format="%Y%m%d_%H%M%S",
    errors="coerce"
)

# Подготавливаем список крупных видеосерий
videos = series[["class", "shooting", "time"]].copy()
videos = videos.rename(columns={
    "shooting": "nearest_series",
    "time": "series_time"
})

# Находим ближайшую серию того же овоща
nearest = pd.merge_asof(
    singles.sort_values("photo_time"),
    videos.sort_values("series_time"),
    left_on="photo_time",
    right_on="series_time",
    by="class",
    direction="nearest"
)

# Вычисляем разницу в минутах
nearest["gap_minutes"] = (
    (nearest["photo_time"] - nearest["series_time"])
    .abs().dt.total_seconds() / 60
)

# Не учитываем видеосерии другого дня
same_day = (
    nearest["photo_time"].dt.date ==
    nearest["series_time"].dt.date
)

nearest.loc[~same_day, "gap_minutes"] = float("nan")

# Сводная таблица
summary_singles = nearest.groupby("class").agg(
    total=("filename", "size"),
    within_1_min=("gap_minutes", lambda x: (x <= 1).sum()),
    within_5_min=("gap_minutes", lambda x: (x <= 5).sum()),
    within_30_min=("gap_minutes", lambda x: (x <= 30).sum())
)

print("ОДИНОЧНЫЕ ФОТОГРАФИИ РЯДОМ С ВИДЕОСЕРИЯМИ")
display(summary_singles)

print("\nПРИМЕРЫ БЛИЗКИХ ПО ВРЕМЕНИ СНИМКОВ")
display(
    nearest.sort_values("gap_minutes")[
        ["class", "filename", "nearest_series", "gap_minutes"]
    ].head(15)
)


In [ ]:
!pip install -q ImageHash

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import imagehash
import pandas as pd

# Папка с исходными изображениями
source_root = Path(dataset_path) / "Dataset" / "Dataset"

# Берём результаты нашего предыдущего анализа
phash_df = original[
    ["class", "filename", "shooting"]
].copy().reset_index(drop=True)

# Вычисляем pHash для каждой фотографии
phashes = []

for vegetable, filename in tqdm(
    zip(phash_df["class"], phash_df["filename"]),
    total=len(phash_df),
    desc="Вычисляем pHash"
):
    file_path = source_root / vegetable / filename

    with Image.open(file_path) as img:
        fingerprint = imagehash.phash(img)
        phashes.append(str(fingerprint))

# Сохраняем полученные отпечатки
phash_df["phash"] = phashes

print("Обработано изображений:", len(phash_df))
display(phash_df.head())

In [ ]:
threshold = 6

# Переводим отпечатки в числа для сравнения
fingerprints = [
    int(h, 16) for h in phash_df["phash"]
]

prefixes = phash_df["shooting"].tolist()
filenames = phash_df["filename"].tolist()

inside_count = 0
between_rows = []

for vegetable, subset in tqdm(
    phash_df.groupby("class"),
    desc="Ищем похожие фотографии"
):
    indexes = subset.index.tolist()

    for pos, i in enumerate(indexes):
        for j in indexes[pos + 1:]:

            distance = (
                fingerprints[i] ^ fingerprints[j]
            ).bit_count()

            if distance <= threshold:

                if prefixes[i] == prefixes[j]:
                    inside_count += 1
                else:
                    between_rows.append({
                        "class": vegetable,
                        "photo_1": filenames[i],
                        "photo_2": filenames[j],
                        "distance": distance
                    })

between_df = pd.DataFrame(
    between_rows,
    columns=["class", "photo_1", "photo_2", "distance"]
)

print("Похожие пары внутри одного префикса:", inside_count)
print("Похожие пары между разными префиксами:", len(between_df))

if not between_df.empty:
    display(
        between_df.sort_values(
            ["distance", "class"]
        ).head(20)
    )

In [ ]:
from collections import Counter
from tqdm.auto import tqdm
import pandas as pd

distance_counts = Counter()
closest_pairs = {}

for vegetable, subset in tqdm(
    phash_df.groupby("class"),
    desc="Сравниваем разные префиксы"
):
    indexes = subset.index.tolist()

    for pos, i in enumerate(indexes):
        for j in indexes[pos + 1:]:

            # Пропускаем один и тот же префикс
            if prefixes[i] == prefixes[j]:
                continue

            distance = (
                fingerprints[i] ^ fingerprints[j]
            ).bit_count()

            distance_counts[distance] += 1

            # Одна запись для каждой пары префиксов
            key = (
                vegetable,
                tuple(sorted((prefixes[i], prefixes[j])))
            )

            if (
                key not in closest_pairs
                or distance < closest_pairs[key]["min_distance"]
            ):
                closest_pairs[key] = {
                    "class": vegetable,
                    "prefix_1": prefixes[i],
                    "prefix_2": prefixes[j],
                    "photo_1": filenames[i],
                    "photo_2": filenames[j],
                    "min_distance": distance
                }

print("Количество похожих пар между префиксами:")

for threshold in [6, 8, 10, 12, 16, 20]:
    count = sum(
        n for d, n in distance_counts.items()
        if d <= threshold
    )
    print(f"pHash <= {threshold}: {count} пар")

if distance_counts:
    print(
        "\nМинимальное расстояние:",
        min(distance_counts)
    )

closest_df = pd.DataFrame(closest_pairs.values())

if not closest_df.empty:
    display(
        closest_df.sort_values("min_distance").head(20)
    )

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Берём 12 наиболее близких пар префиксов
candidates = (
    closest_df
    .sort_values("min_distance")
    .head(12)
    .reset_index(drop=True)
)

fig, axes = plt.subplots(
    len(candidates), 2,
    figsize=(12, 3 * len(candidates))
)

for i, row in candidates.iterrows():

    vegetable = row["class"]

    path1 = source_root / vegetable / row["photo_1"]
    path2 = source_root / vegetable / row["photo_2"]

    with Image.open(path1) as img:
        axes[i, 0].imshow(img.convert("RGB"))

    with Image.open(path2) as img:
        axes[i, 1].imshow(img.convert("RGB"))

    axes[i, 0].set_title(
        f"{vegetable}: {row['photo_1']}",
        fontsize=9
    )

    axes[i, 1].set_title(
        f"{vegetable}: {row['photo_2']}",
        fontsize=9
    )

    axes[i, 0].set_ylabel(
        f"pHash = {row['min_distance']}",
        fontsize=10
    )

    axes[i, 0].axis("off")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Составляем таблицу префиксов
timeline = (
    phash_df.groupby(["class", "shooting"])
    .agg(n_images=("filename", "size"))
    .reset_index()
)

# Получаем предполагаемое время из имени префикса
timeline["time"] = pd.to_datetime(
    timeline["shooting"],
    format="%Y%m%d_%H%M%S",
    errors="coerce"
)

# Проверяем, что время получилось извлечь
print("Не распознано дат:", timeline["time"].isna().sum())

timeline["date"] = timeline["time"].dt.date

# Сортируем по классу и времени
timeline = timeline.sort_values(
    ["class", "time"]
).reset_index(drop=True)

# Разница между соседними префиксами в минутах
timeline["gap_min"] = (
    timeline.groupby(["class", "date"])["time"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

# Новый временной блок, если разрыв больше 5 минут
timeline["new_block"] = (
    timeline["gap_min"].isna()
    | (timeline["gap_min"] > 5)
)

timeline["block"] = (
    timeline.groupby(["class", "date"])["new_block"]
    .cumsum()
)

# Сводка по временным блокам
blocks = (
    timeline.groupby(["class", "date", "block"])
    .agg(
        start=("time", "min"),
        end=("time", "max"),
        prefixes=("shooting", "size"),
        images=("n_images", "sum")
    )
    .reset_index()
)

print("Предварительных временных блоков:", len(blocks))

display(
    blocks[
        ["class", "start", "end", "prefixes", "images"]
    ]
)

In [ ]:
# Берём готовые данные из предыдущей проверки
all_timeline = timeline[
    ["class", "shooting", "time", "n_images"]
].copy()

# Сортируем все префиксы по времени,
# независимо от класса овоща
all_timeline = all_timeline.sort_values(
    "time"
).reset_index(drop=True)

# Рассчитываем временной промежуток
# между соседними префиксами
all_timeline["gap_min"] = (
    all_timeline["time"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

# Новый блок начинается после разрыва > 5 минут
all_timeline["new_block"] = (
    all_timeline["gap_min"].isna()
    | (all_timeline["gap_min"] > 5)
)

# Присваиваем номер каждому временному блоку
all_timeline["time_block_id"] = (
    all_timeline["new_block"].cumsum()
)

# Создаём итоговую сводку
global_blocks = (
    all_timeline.groupby("time_block_id")
    .agg(
        start=("time", "min"),
        end=("time", "max"),
        classes=("class", lambda x: ", ".join(sorted(x.unique()))),
        n_classes=("class", "nunique"),
        prefixes=("shooting", "size"),
        images=("n_images", "sum")
    )
    .reset_index()
)

print("Общих временных блоков:", len(global_blocks))
print("Всего изображений:", global_blocks["images"].sum())

display(global_blocks)

In [ ]:
# Количество изображений каждого класса
# в каждом временном блоке

class_distribution = all_timeline.pivot_table(
    index="time_block_id",
    columns="class",
    values="n_images",
    aggfunc="sum",
    fill_value=0
)

# Добавляем итог по каждому временному блоку
class_distribution["Total"] = class_distribution.sum(axis=1)

# Добавляем итог по каждому классу
class_distribution.loc["Total"] = class_distribution.sum()

display(class_distribution)


In [ ]:
# Создаём соответствие префикса и группы съёмки
group_map = all_timeline[
    ["class", "shooting", "time"]
].copy()

# Каждая дата становится отдельной группой съёмки
group_map["shooting_group"] = (
    "S_" + group_map["time"].dt.strftime("%Y%m%d")
)

# Добавляем группу к каждой фотографии
phash_df = phash_df.drop(
    columns=["shooting_group"],
    errors="ignore"
).merge(
    group_map[["class", "shooting", "shooting_group"]],
    on=["class", "shooting"],
    how="left",
    validate="many_to_one"
)

# Проверяем результат
print("Фотографий:", len(phash_df))
print(
    "Групп съёмки:",
    phash_df["shooting_group"].nunique()
)
print(
    "Без группы:",
    phash_df["shooting_group"].isna().sum()
)

display(
    phash_df.groupby("shooting_group")
    .size()
    .reset_index(name="images")
)

In [ ]:
print("Столбцы phash_df:")
print(phash_df.columns.tolist())

print("\nПапка датасета:")
print(source_root)

print("\nПример двух записей:")
display(phash_df.head(2).T)


In [ ]:
from pathlib import Path
import cv2
import pandas as pd
import matplotlib.pyplot as plt

# 1. Находим изображения, не изменяя исходные файлы
image_index = {}

for path in Path(source_root).rglob("*"):
    if path.is_file() and path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
        key = (path.parent.name.lower(), path.name)
        image_index.setdefault(key, []).append(path)

# 2. Берём по 3 случайных фотографии из каждого класса
blur_sample = pd.concat([
    group.sample(n=min(3, len(group)), random_state=42)
    for _, group in phash_df.groupby("class")
]).reset_index(drop=True)

# 3. Рассчитываем показатель резкости
def calculate_sharpness(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    h, w = gray.shape
    new_h = max(1, round(h * 256 / w))

    gray = cv2.resize(
        gray, (256, new_h),
        interpolation=cv2.INTER_AREA
    )

    return cv2.Laplacian(gray, cv2.CV_64F).var()

# 4. Показываем фотографии и показатели
fig, axes = plt.subplots(6, 3, figsize=(12, 19))

scores = []

for i, row in blur_sample.iterrows():
    key = (row["class"].lower(), row["filename"])
    paths = image_index.get(key, [])

    ax = axes.flat[i]

    if not paths:
        scores.append(float("nan"))
        ax.text(0.5, 0.5, "Файл не найден",
                ha="center", va="center")
        ax.axis("off")
        continue

    image = cv2.imread(str(paths[0]))

    if image is None:
        scores.append(float("nan"))
        ax.text(0.5, 0.5, "Ошибка чтения",
                ha="center", va="center")
        ax.axis("off")
        continue

    score = calculate_sharpness(image)
    scores.append(round(score, 2))

    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(
        f"{row['class']} | Резкость: {score:.1f}",
        fontsize=10
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

blur_sample["sharpness"] = scores

display(
    blur_sample[
        ["class", "filename", "shooting_group", "sharpness"]
    ]
)

print("Успешно проверено:",
      blur_sample["sharpness"].notna().sum(),
      "из", len(blur_sample))

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Два проблемных кадра, которые ты нашёл
bad_examples = [
    "20251113_210803_frame_4.jpg",
    "20251113_210803_frame_10.jpg"
]

# Три фотографии Carrot из предыдущей проверки
normal_examples = blur_sample[
    blur_sample["class"] == "Carrot"
]["filename"].tolist()

test_names = bad_examples + normal_examples

fig, axes = plt.subplots(1, len(test_names), figsize=(18, 4))

for ax, name in zip(axes, test_names):

    # Ищем файл в уже созданном индексе
    matches = [
        path
        for (cls, filename), paths in image_index.items()
        if filename == name
        for path in paths
    ]

    if not matches:
        ax.set_title("Файл не найден")
        ax.axis("off")
        continue

    image = cv2.imread(str(matches[0]))

    if image is None:
        ax.set_title("Ошибка чтения")
        ax.axis("off")
        continue

    # Процент почти белых пикселей
    white_mask = np.all(image >= 245, axis=2)
    white_percent = white_mask.mean() * 100

    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(
        f"{name}\nПочти белые: {white_percent:.1f}%",
        fontsize=9
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import hashlib
import numpy as np
import cv2
import pandas as pd

exposure_df = phash_df.copy()

white_scores = []
statuses = []

for cls, filename in zip(
    exposure_df["class"],
    exposure_df["filename"]
):
    paths = image_index.get(
        (cls.lower(), filename), []
    )

    if not paths:
        white_scores.append(np.nan)
        statuses.append("not_found")
        continue

    # Если есть несколько файлов с одинаковым именем,
    # проверяем, совпадает ли их содержимое
    if len(paths) > 1:
        hashes = {
            hashlib.sha256(p.read_bytes()).hexdigest()
            for p in paths
        }

        if len(hashes) > 1:
            white_scores.append(np.nan)
            statuses.append("ambiguous")
            continue

    image = cv2.imread(str(paths[0]))

    if image is None:
        white_scores.append(np.nan)
        statuses.append("read_error")
        continue

    white_mask = np.all(image >= 245, axis=2)
    white_percent = white_mask.mean() * 100

    white_scores.append(round(white_percent, 2))
    statuses.append("ok")

exposure_df["white_pct"] = white_scores
exposure_df["exposure_status"] = statuses

# Общая статистика по классам
exposure_summary = exposure_df.groupby("class").agg(
    total=("filename", "size"),
    checked=("white_pct", "count"),
    above_30=("white_pct", lambda x: (x >= 30).sum()),
    above_50=("white_pct", lambda x: (x >= 50).sum()),
    max_white=("white_pct", "max")
)

print("Результаты проверки экспозиции:")
display(exposure_summary.round(2))

print("\nСтатусы обработки:")
print(exposure_df["exposure_status"].value_counts())

print("\n12 фотографий с наибольшей долей белых пикселей:")
display(
    exposure_df.nlargest(12, "white_pct")[
        ["class", "filename", "shooting_group", "white_pct"]
    ]
)

In [ ]:
import matplotlib.pyplot as plt
import cv2
import math

# Все подозрительные фотографии
suspects = exposure_df[
    exposure_df["white_pct"] >= 30
].sort_values("white_pct", ascending=False)

n = len(suspects)
cols = 4
rows = math.ceil(n / cols)

fig, axes = plt.subplots(
    rows, cols,
    figsize=(16, rows * 4)
)

axes = axes.flatten()

for i, (_, row) in enumerate(suspects.iterrows()):

    key = (row["class"].lower(), row["filename"])
    paths = image_index.get(key, [])

    ax = axes[i]

    if paths:
        image = cv2.imread(str(paths[0]))

        if image is not None:
            ax.imshow(
                cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            )
        else:
            ax.text(0.5, 0.5, "Ошибка чтения",
                    ha="center", va="center")
    else:
        ax.text(0.5, 0.5, "Файл не найден",
                ha="center", va="center")

    ax.set_title(
        f"{row['class']} | {row['white_pct']:.1f}%\n"
        f"{row['filename']}",
        fontsize=9
    )
    ax.axis("off")

# Скрываем пустые ячейки
for j in range(n, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Список явно пересвеченных кадров Carrot
carrot_bad = [
    f"20251113_210803_frame_{i}.jpg"
    for i in range(14)
]

# Список подозрительных кадров Squash
squash_review = [
    f"20251117_134731_frame_{i}.jpg"
    for i in range(106, 110)
]

# Находим только визуально проверенные фотографии
carrot_mask = (
    (exposure_df["class"] == "Carrot") &
    exposure_df["filename"].isin(carrot_bad)
)

squash_mask = (
    (exposure_df["class"] == "Squash") &
    exposure_df["filename"].isin(squash_review)
)

quality_flags_df = exposure_df.loc[
    carrot_mask | squash_mask,
    ["class", "filename", "shooting", "shooting_group", "white_pct"]
].copy()

# Фиксируем предварительное решение
quality_flags_df["quality_decision"] = "manual_review"

quality_flags_df.loc[
    quality_flags_df["class"] == "Carrot",
    "quality_decision"
] = "exclude_candidate"

# Проверяем полноту отметок
assert carrot_mask.sum() == 14
assert squash_mask.sum() == 4

print("Результаты визуальной проверки:")
print(quality_flags_df["quality_decision"].value_counts())

display(
    quality_flags_df.sort_values(
        ["class", "filename"]
    )
)

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Отмечаем все 18 кадров как кандидатов на исключение
quality_flags_df["quality_decision"] = "exclude_candidate"

# Создаём таблицу для анализа темноты
dark_df = exposure_df[
    ["class", "filename", "shooting", "shooting_group"]
].copy()

dark_scores = []
brightness_scores = []

for cls, filename in zip(
    dark_df["class"], dark_df["filename"]
):
    paths = image_index.get((cls.lower(), filename), [])

    image = cv2.imread(str(paths[0])) if paths else None

    if image is None:
        dark_scores.append(np.nan)
        brightness_scores.append(np.nan)
        continue

    # Переводим изображение в оттенки серого
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Доля очень тёмных пикселей
    dark_pct = (gray <= 40).mean() * 100

    # Яркость, ниже которой находятся 90% пикселей
    p90 = np.percentile(gray, 90)

    dark_scores.append(round(dark_pct, 2))
    brightness_scores.append(round(p90, 2))

dark_df["dark_pct"] = dark_scores
dark_df["brightness_p90"] = brightness_scores

print("Проверено:", dark_df["brightness_p90"].notna().sum())
print("Кандидатов на исключение:", len(quality_flags_df))

# Находим 12 самых тёмных изображений
darkest = dark_df.nsmallest(12, "brightness_p90")

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for ax, (_, row) in zip(axes.flat, darkest.iterrows()):

    paths = image_index.get(
        (row["class"].lower(), row["filename"]), []
    )

    image = cv2.imread(str(paths[0]))

    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

    ax.set_title(
        f"{row['class']} | P90: {row['brightness_p90']:.1f}\n"
        f"Тёмных пикселей: {row['dark_pct']:.1f}%",
        fontsize=10
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

display(
    darkest[
        ["class", "filename", "dark_pct", "brightness_p90"]
    ]
)

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

classes_to_check = ["Carrot", "Capsicum", "Squash"]

color_results = []

fig, axes = plt.subplots(3, 6, figsize=(18, 11))

for row_id, cls in enumerate(classes_to_check):

    # Две фотографии с низким показателем общей яркости
    low_examples = (
        dark_df[dark_df["class"] == cls]
        .nsmallest(2, "brightness_p90")["filename"]
        .tolist()
    )

    # Одна фотография из первоначальной выборки
    normal_example = (
        blur_sample[blur_sample["class"] == cls]
        ["filename"].iloc[0]
    )

    examples = low_examples + [normal_example]

    for j, filename in enumerate(examples):

        paths = image_index.get(
            (cls.lower(), filename), []
        )

        ax_img = axes[row_id, j * 2]
        ax_mask = axes[row_id, j * 2 + 1]

        image = cv2.imread(str(paths[0])) if paths else None

        if image is None:
            ax_img.set_title("Не удалось открыть")
            ax_img.axis("off")
            ax_mask.axis("off")
            continue

        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

        h = hsv[:, :, 0]
        s = hsv[:, :, 1]
        v = hsv[:, :, 2]

        # Тёмные пиксели
        dark_mask = v < 80

        # Оранжевые пиксели
        orange_mask = (
            (h >= 5) & (h <= 25) &
            (s >= 80) & (v >= 70)
        )

        # Зелёные пиксели
        green_mask = (
            (h >= 30) & (h <= 95) &
            (s >= 35) & (v >= 55)
        )

        # Выбираем цвет, характерный для класса
        if cls == "Carrot":
            product_mask = orange_mask
        else:
            product_mask = green_mask

        color_pct = product_mask.mean() * 100
        dark_pct = dark_mask.mean() * 100

        color_results.append({
            "class": cls,
            "filename": filename,
            "product_color_pct": round(color_pct, 2),
            "dark_pct": round(dark_pct, 2)
        })

        # Оригинальная фотография
        ax_img.imshow(
            cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        )
        ax_img.set_title(
            f"{cls}\nОригинал",
            fontsize=10
        )

        # Маска найденного цвета
        ax_mask.imshow(
            product_mask,
            cmap="gray",
            vmin=0,
            vmax=1
        )
        ax_mask.set_title(
            f"Цвет: {color_pct:.1f}%\n"
            f"Тёмные: {dark_pct:.1f}%",
            fontsize=10
        )

        ax_img.axis("off")
        ax_mask.axis("off")

plt.tight_layout()
plt.show()

color_test_df = pd.DataFrame(color_results)

display(color_test_df)

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

position_results = []

fig, axes = plt.subplots(3, 3, figsize=(12, 13))

for ax, (_, row) in zip(
    axes.flat, color_test_df.iterrows()
):
    cls = row["class"]
    filename = row["filename"]

    paths = image_index.get(
        (cls.lower(), filename), []
    )

    image = cv2.imread(str(paths[0]))
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Выделяем характерный цвет
    if cls == "Carrot":
        mask = cv2.inRange(
            hsv, (5, 80, 70), (25, 255, 255)
        )
    else:
        mask = cv2.inRange(
            hsv, (30, 35, 55), (95, 255, 255)
        )

    # Убираем мелкий цветовой шум
    kernel = np.ones((3, 3), np.uint8)

    cleaned = cv2.morphologyEx(
        mask, cv2.MORPH_OPEN, kernel
    )

    # Ищем отдельные связные области
    count, labels, stats, centers = (
        cv2.connectedComponentsWithStats(
            cleaned, connectivity=8
        )
    )

    result = image.copy()
    H, W = image.shape[:2]

    largest_pct = 0
    touches_border = False

    if count > 1:
        # Находим крупнейшую область, кроме фона
        idx = 1 + np.argmax(
            stats[1:, cv2.CC_STAT_AREA]
        )

        x, y, w, h, area = stats[idx]

        largest_pct = area / (H * W) * 100

        # Проверяем близость к краям изображения
        touches_border = (
            x <= 2 or y <= 2 or
            x + w >= W - 2 or
            y + h >= H - 2
        )

        # Рисуем прямоугольник вокруг области
        cv2.rectangle(
            result,
            (x, y),
            (x + w, y + h),
            (0, 255, 255),
            2
        )

    position_results.append({
        "class": cls,
        "filename": filename,
        "largest_color_pct": round(largest_pct, 2),
        "touches_border": touches_border
    })

    ax.imshow(
        cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
    )

    ax.set_title(
        f"{cls}\n"
        f"Область: {largest_pct:.1f}% | "
        f"Край: {touches_border}",
        fontsize=10
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

position_df = pd.DataFrame(position_results)

display(position_df)

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

target_classes = ["Carrot", "Capsicum", "Squash"]

records = []

for _, row in phash_df[
    phash_df["class"].isin(target_classes)
].iterrows():

    cls = row["class"]
    filename = row["filename"]

    paths = image_index.get(
        (cls.lower(), filename), []
    )

    image = cv2.imread(str(paths[0])) if paths else None

    if image is None:
        continue

    H, W = image.shape[:2]
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    if cls == "Carrot":
        mask = cv2.inRange(
            hsv, (5, 80, 70), (25, 255, 255)
        )
    else:
        mask = cv2.inRange(
            hsv, (30, 35, 55), (95, 255, 255)
        )

    cleaned = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        np.ones((3, 3), np.uint8)
    )

    count, labels, stats, centers = (
        cv2.connectedComponentsWithStats(
            cleaned, connectivity=8
        )
    )

    largest_pct = 0
    touches_border = False

    if count > 1:
        idx = 1 + np.argmax(
            stats[1:, cv2.CC_STAT_AREA]
        )

        x, y, w, h, area = stats[idx]

        largest_pct = area / (H * W) * 100

        touches_border = (
            x <= 2 or y <= 2 or
            x + w >= W - 2 or
            y + h >= H - 2
        )

    records.append({
        "class": cls,
        "filename": filename,
        "shooting": row["shooting"],
        "shooting_group": row["shooting_group"],
        "largest_color_pct": round(largest_pct, 2),
        "touches_border": touches_border
    })

# Сохраняем результаты в отдельной таблице
composition_df = pd.DataFrame(records)

# Помечаем подозрительные кадры
composition_df["composition_suspect"] = (
    (composition_df["largest_color_pct"] < 12) &
    composition_df["touches_border"]
)

print("Проверено изображений:", len(composition_df))

print("\nПодозрительные изображения по классам:")
display(
    composition_df.groupby("class")[
        "composition_suspect"
    ].agg(["sum", "count"])
)

# Берём 12 подозрительных изображений,
# имеющих наименьшую цветовую область
suspects = (
    composition_df[
        composition_df["composition_suspect"]
    ]
    .sort_values("largest_color_pct")
    .head(12)
)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for ax in axes.flat:
    ax.axis("off")

for ax, (_, row) in zip(axes.flat, suspects.iterrows()):

    paths = image_index.get(
        (row["class"].lower(), row["filename"]), []
    )

    image = cv2.imread(str(paths[0]))

    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

    ax.set_title(
        f"{row['class']} | {row['largest_color_pct']:.1f}%\n"
        f"{row['filename']}",
        fontsize=9
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Уже отмеченные 18 фотографий
known_flags = set(
    zip(
        quality_flags_df["class"],
        quality_flags_df["filename"]
    )
)

# Все 90 подозрительных фотографий
composition_review_df = composition_df[
    composition_df["composition_suspect"]
].copy()

# Проверяем, какие из них уже были отмечены
composition_review_df["already_flagged"] = [
    (cls, filename) in known_flags
    for cls, filename in zip(
        composition_review_df["class"],
        composition_review_df["filename"]
    )
]

print("Всего подозрительных:",
      len(composition_review_df))

print("Уже отмечены ранее:",
      composition_review_df["already_flagged"].sum())

# Оставляем только ещё не отмеченные изображения
new_composition_suspects = composition_review_df[
    ~composition_review_df["already_flagged"]
].copy()

print("Новых подозрительных:",
      len(new_composition_suspects))

# Смотрим распределение по видеосериям
shooting_summary = (
    new_composition_suspects
    .groupby(["class", "shooting"])
    .size()
    .reset_index(name="suspect_count")
    .sort_values("suspect_count", ascending=False)
)

print("\nРаспределение по видеосериям:")
display(shooting_summary)

# Отдельно проверяем одиночные фотографии
single_suspects = new_composition_suspects[
    new_composition_suspects["shooting"].isna()
]

print("\nПодозрительных одиночных фотографий:",
      len(single_suspects))

In [ ]:
import matplotlib.pyplot as plt
import cv2
import math

# Выбираем нужную видеосерию
review = new_composition_suspects[
    (new_composition_suspects["class"] == "Capsicum") &
    (new_composition_suspects["shooting"] == "20251117_134205")
].copy()

# Извлекаем номера кадров
review["frame_num"] = (
    review["filename"]
    .str.extract(r"_frame_(\d+)\.")[0]
    .astype(int)
)

# Располагаем кадры по порядку
review = review.sort_values("frame_num")

print("Подозрительных кадров:", len(review))
print("Номера кадров:", review["frame_num"].tolist())

# Отображаем фотографии
cols = 5
rows = math.ceil(len(review) / cols)

fig, axes = plt.subplots(
    rows, cols,
    figsize=(18, rows * 4)
)

axes = axes.flatten()

for ax in axes:
    ax.axis("off")

for ax, (_, row) in zip(axes, review.iterrows()):

    paths = image_index.get(
        (row["class"].lower(), row["filename"]), []
    )

    if not paths:
        ax.set_title("Файл не найден")
        continue

    image = cv2.imread(str(paths[0]))

    if image is None:
        ax.set_title("Ошибка чтения")
        continue

    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

    ax.set_title(
        f"Кадр №{row['frame_num']}\n"
        f"Цветовая область: {row['largest_color_pct']:.1f}%",
        fontsize=10
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Добавляем причину для ранее отмеченных 18 кадров
if "quality_reason" not in quality_flags_df.columns:
    quality_flags_df["quality_reason"] = "severe_overexposure"

# Берём 30 визуально проверенных кадров Capsicum
new_flags = review[
    ["class", "filename", "shooting", "shooting_group"]
].drop_duplicates().copy()

# Добавляем ранее рассчитанную долю белых пикселей
new_flags = new_flags.merge(
    exposure_df[["class", "filename", "white_pct"]],
    on=["class", "filename"],
    how="left",
    validate="one_to_one"
)

new_flags["quality_decision"] = "exclude_candidate"
new_flags["quality_reason"] = "poor_composition_or_occlusion"

# Добавляем к существующим отметкам
quality_flags_df = pd.concat(
    [quality_flags_df, new_flags],
    ignore_index=True
).drop_duplicates(
    subset=["class", "filename"],
    keep="last"
)

print("Всего кандидатов на исключение:",
      len(quality_flags_df))

print("\nПо причинам:")
display(
    quality_flags_df["quality_reason"].value_counts()
)

print("\nПо классам:")
display(
    quality_flags_df.groupby("class").size()
)

In [ ]:
import matplotlib.pyplot as plt
import cv2
import math

# Выбираем 14 подозрительных фотографий Squash
squash_comp_review = new_composition_suspects[
    (new_composition_suspects["class"] == "Squash") &
    (new_composition_suspects["shooting"] == "20260101_134905")
].copy()

# Определяем номер кадра
squash_comp_review["frame_num"] = (
    squash_comp_review["filename"]
    .str.extract(r"_frame_(\d+)\.")[0]
    .astype(int)
)

# Сортируем по порядку съёмки
squash_comp_review = squash_comp_review.sort_values("frame_num")

print("Количество кадров:", len(squash_comp_review))
print("Номера кадров:", squash_comp_review["frame_num"].tolist())

# Показываем фотографии
cols = 4
rows = math.ceil(len(squash_comp_review) / cols)

fig, axes = plt.subplots(
    rows, cols,
    figsize=(16, rows * 4)
)

for ax in axes.flat:
    ax.axis("off")

for ax, (_, row) in zip(
    axes.flat,
    squash_comp_review.iterrows()
):
    paths = image_index.get(
        (row["class"].lower(), row["filename"]), []
    )

    if not paths:
        ax.set_title("Файл не найден")
        continue

    image = cv2.imread(str(paths[0]))

    if image is None:
        ax.set_title("Ошибка чтения")
        continue

    ax.imshow(
        cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    )

    ax.set_title(
        f"Кадр №{row['frame_num']}\n"
        f"Цветовая область: {row['largest_color_pct']:.1f}%",
        fontsize=10
    )

plt.tight_layout()
plt.show()

In [ ]:
# Сохраняем решения по всем 14 кадрам
squash_review_log = squash_comp_review[
    ["class", "filename", "shooting",
     "shooting_group", "frame_num"]
].copy()

# Кадры 150–159 — кандидаты на исключение
bad_mask = squash_review_log["frame_num"].between(150, 159)

squash_review_log["quality_decision"] = np.where(
    bad_mask,
    "exclude_candidate",
    "keep_after_visual_review"
)

squash_review_log["quality_reason"] = np.where(
    bad_mask,
    "mostly_out_of_frame",
    "visible_product_false_positive"
)

# Проверяем количество
assert len(squash_review_log) == 14
assert bad_mask.sum() == 10

# Добавляем только 10 кандидатов на исключение
new_flags = squash_review_log[
    squash_review_log["quality_decision"] == "exclude_candidate"
].copy()

new_flags = new_flags.merge(
    exposure_df[["class", "filename", "white_pct"]],
    on=["class", "filename"],
    how="left",
    validate="one_to_one"
)

quality_flags_df = pd.concat(
    [
        quality_flags_df,
        new_flags[
            ["class", "filename", "shooting",
             "shooting_group", "white_pct",
             "quality_decision", "quality_reason"]
        ]
    ],
    ignore_index=True
).drop_duplicates(
    subset=["class", "filename"],
    keep="last"
)

print("Результаты текущей проверки:")
print(squash_review_log["quality_decision"].value_counts())

print("\nВсего кандидатов на исключение:",
      len(quality_flags_df))

print("\nРаспределение кандидатов по классам:")
display(quality_flags_df.groupby("class").size())

In [ ]:
# Выбираем 12 подозрительных кадров нужной видеосерии
squash_review_2 = new_composition_suspects[
    (new_composition_suspects["class"] == "Squash") &
    (new_composition_suspects["shooting"] == "20260101_133006")
].copy()

# Извлекаем номера кадров
squash_review_2["frame_num"] = (
    squash_review_2["filename"]
    .str.extract(r"_frame_(\d+)\.")[0]
    .astype(int)
)

# Сортируем по порядку съёмки
squash_review_2 = squash_review_2.sort_values("frame_num")

print("Количество кадров:", len(squash_review_2))
print("Номера кадров:", squash_review_2["frame_num"].tolist())

# Показываем фотографии
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for ax in axes.flat:
    ax.axis("off")

for ax, (_, row) in zip(axes.flat, squash_review_2.iterrows()):

    paths = image_index.get(
        (row["class"].lower(), row["filename"]), []
    )

    if not paths:
        ax.set_title("Файл не найден")
        continue

    image = cv2.imread(str(paths[0]))

    if image is None:
        ax.set_title("Ошибка чтения")
        continue

    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

    ax.set_title(
        f"Кадр №{row['frame_num']}\n"
        f"Цветовая область: {row['largest_color_pct']:.1f}%",
        fontsize=10
    )

plt.tight_layout()
plt.show()

In [ ]:
# Фиксируем решение по 12 новым кадрам
squash_review_2_log = squash_review_2[
    ["class", "filename", "shooting",
     "shooting_group", "frame_num"]
].copy()

squash_review_2_log["quality_decision"] = (
    "keep_after_visual_review"
)

squash_review_2_log["quality_reason"] = (
    "color_segmentation_false_positive"
)

# Объединяем с предыдущими 14 проверенными кадрами Squash
composition_review_log = pd.concat(
    [squash_review_log, squash_review_2_log],
    ignore_index=True
).drop_duplicates(
    subset=["class", "filename"],
    keep="last"
)

print("Результаты двух проверок Squash:")
print(composition_review_log["quality_decision"].value_counts())

print("\nВсего кандидатов на исключение:",
      len(quality_flags_df))

In [ ]:
import matplotlib.pyplot as plt
import cv2
import math

# Выбираем подозрительные кадры Carrot
carrot_review = new_composition_suspects[
    (new_composition_suspects["class"] == "Carrot") &
    (new_composition_suspects["shooting"] == "20260101_135022")
].copy()

# Извлекаем номера кадров
carrot_review["frame_num"] = (
    carrot_review["filename"]
    .str.extract(r"_frame_(\d+)\.")[0]
    .astype(int)
)

carrot_review = carrot_review.sort_values("frame_num")

print("Количество кадров:", len(carrot_review))
print("Номера кадров:", carrot_review["frame_num"].tolist())

# Выводим фотографии
cols = 4
rows = math.ceil(len(carrot_review) / cols)

fig, axes = plt.subplots(
    rows, cols,
    figsize=(16, rows * 4)
)

for ax in axes.flat:
    ax.axis("off")

for ax, (_, row) in zip(axes.flat, carrot_review.iterrows()):

    paths = image_index.get(
        (row["class"].lower(), row["filename"]), []
    )

    if not paths:
        ax.set_title("Файл не найден")
        continue

    image = cv2.imread(str(paths[0]))

    if image is None:
        ax.set_title("Ошибка чтения")
        continue

    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

    ax.set_title(
        f"Кадр №{row['frame_num']}\n"
        f"Оранжевая область: {row['largest_color_pct']:.1f}%",
        fontsize=10
    )

plt.tight_layout()
plt.show()

In [ ]:
# Фиксируем решение по Carrot
carrot_review_log = carrot_review[
    ["class", "filename", "shooting",
     "shooting_group", "frame_num"]
].copy()

# Проверяем, что работаем с нужными 11 кадрами
expected_frames = {15, 25, 26, 27, 28, 29, 30,
                   163, 164, 165, 166}

assert set(carrot_review_log["frame_num"]) == expected_frames
assert len(carrot_review_log) == 11

carrot_review_log["quality_decision"] = "exclude_candidate"

# Разделяем причины исключения
carrot_review_log["quality_reason"] = np.where(
    carrot_review_log["frame_num"] < 163,
    "mostly_out_of_frame_or_occluded",
    "heavily_cropped_carrots"
)

# Добавляем показатель пересвета из предыдущего анализа
new_flags = carrot_review_log.merge(
    exposure_df[["class", "filename", "white_pct"]],
    on=["class", "filename"],
    how="left",
    validate="one_to_one"
)

# Обновляем общую таблицу кандидатов
quality_flags_df = pd.concat(
    [
        quality_flags_df,
        new_flags[
            ["class", "filename", "shooting",
             "shooting_group", "white_pct",
             "quality_decision", "quality_reason"]
        ]
    ],
    ignore_index=True
).drop_duplicates(
    subset=["class", "filename"],
    keep="last"
)

print("Всего кандидатов на исключение:",
      len(quality_flags_df))

print("\nПо классам:")
display(quality_flags_df.groupby("class").size())

print("\nПо причинам:")
display(quality_flags_df["quality_reason"].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import cv2
import math
import pandas as pd

# Уже рассмотренные изображения
excluded_keys = set(zip(
    quality_flags_df["class"],
    quality_flags_df["filename"]
))

reviewed_keys = set(zip(
    composition_review_log["class"],
    composition_review_log["filename"]
))

all_checked_keys = excluded_keys | reviewed_keys

# Оставшиеся подозрительные фотографии
remaining_review = new_composition_suspects[
    ~new_composition_suspects.apply(
        lambda row: (
            row["class"], row["filename"]
        ) in all_checked_keys,
        axis=1
    )
].copy()

# Исправление: разрешаем отсутствие номера кадра
remaining_review["frame_num"] = pd.to_numeric(
    remaining_review["filename"].str.extract(
        r"_frame_(\d+)\."
    )[0],
    errors="coerce"
).astype("Int64")

remaining_review = remaining_review.sort_values(
    ["class", "shooting", "frame_num"],
    na_position="last"
)

print("Осталось проверить:", len(remaining_review))

print("\nФайлы без номера кадра:")
display(
    remaining_review[
        remaining_review["frame_num"].isna()
    ][["class", "filename", "shooting"]]
)

# Отображаем фотографии
n = len(remaining_review)
cols = 5
rows = math.ceil(n / cols)

fig, axes = plt.subplots(
    rows, cols,
    figsize=(18, rows * 4),
    squeeze=False
)

for ax in axes.flat:
    ax.axis("off")

for ax, (_, row) in zip(
    axes.flat,
    remaining_review.iterrows()
):
    paths = image_index.get(
        (row["class"].lower(), row["filename"]), []
    )

    image = cv2.imread(str(paths[0])) if paths else None

    if image is None:
        ax.set_title("Файл не найден")
        continue

    ax.imshow(
        cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    )

    if pd.notna(row["frame_num"]):
        label = f"Кадр №{int(row['frame_num'])}"
    else:
        label = "Без номера кадра"

    ax.set_title(
        f"{row['class']} | {label}\n"
        f"Цвет: {row['largest_color_pct']:.1f}%\n"
        f"{row['filename']}",
        fontsize=9
    )

plt.tight_layout()
plt.show()

In [ ]:
# Единственная фотография, которую сохраняем
keep_filename = "20251113_205619.jpg"

# Проверяем, что рассматриваем именно 15 изображений
assert len(remaining_review) == 15

keep_mask = (
    (remaining_review["class"] == "Capsicum") &
    (remaining_review["filename"] == keep_filename)
)

assert keep_mask.sum() == 1

# Остальные 14 — кандидаты на исключение
new_flags = remaining_review.loc[
    ~keep_mask,
    ["class", "filename", "shooting", "shooting_group"]
].copy()

new_flags = new_flags.merge(
    exposure_df[["class", "filename", "white_pct"]],
    on=["class", "filename"],
    how="left",
    validate="one_to_one"
)

new_flags["quality_decision"] = "exclude_candidate"
new_flags["quality_reason"] = "poor_composition_or_occlusion"

# Отдельно отмечаем визуально обнаруженный пересвет
flash_mask = (
    (new_flags["class"] == "Squash") &
    (new_flags["shooting"] == "20251113_210502")
)

new_flags.loc[
    flash_mask, "quality_reason"
] = "visual_overexposure"

# Добавляем к ранее отмеченным изображениям
quality_flags_df = pd.concat(
    [quality_flags_df, new_flags],
    ignore_index=True
).drop_duplicates(
    subset=["class", "filename"],
    keep="last"
)

print("Всего кандидатов:", len(quality_flags_df))

print("\nПо классам:")
display(quality_flags_df.groupby("class").size())

print("\nПересвеченные Squash, пропущенные прежней проверкой:")
display(
    new_flags.loc[
        flash_mask,
        ["filename", "white_pct", "quality_reason"]
    ]
)

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Три фотографии с сильным пересветом
bad_images = [
    "20251113_210502_frame_1.jpg",
    "20251113_210502_frame_2.jpg",
    "20251113_210502_frame_6.jpg"
]

# Три нормальные фотографии Squash
good_images = [
    "20260101_133006_frame_181.jpg",
    "20260101_134905_frame_315.jpg",
    "20251117_134731_frame_170.jpg"
]

examples = (
    [("Пересвет", name) for name in bad_images] +
    [("Нормальная", name) for name in good_images]
)

results = []

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax, (category, filename) in zip(
    axes.flat, examples
):
    paths = image_index.get(
        ("squash", filename), []
    )

    if not paths:
        raise FileNotFoundError(filename)

    image = cv2.imread(str(paths[0]))

    if image is None:
        raise ValueError(f"Ошибка чтения: {filename}")

    hsv = cv2.cvtColor(
        image, cv2.COLOR_BGR2HSV
    )

    s = hsv[:, :, 1]
    v = hsv[:, :, 2]

    # Старый метод: почти белые RGB-пиксели
    white_rgb_pct = (
        np.all(image >= 245, axis=2).mean() * 100
    )

    # Новый показатель: очень яркие пиксели
    bright_v_pct = (
        (v >= 245).mean() * 100
    )

    # Очень яркие пиксели с невысокой насыщенностью
    bright_low_sat_pct = (
        ((v >= 245) & (s <= 100)).mean() * 100
    )

    results.append({
        "category": category,
        "filename": filename,
        "white_rgb_pct": round(white_rgb_pct, 2),
        "bright_v_pct": round(bright_v_pct, 2),
        "bright_low_sat_pct": round(
            bright_low_sat_pct, 2
        )
    })

    ax.imshow(
        cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    )

    ax.set_title(
        f"{category}\n"
        f"RGB белые: {white_rgb_pct:.1f}%\n"
        f"HSV яркие: {bright_v_pct:.1f}%\n"
        f"Яркие + слабая насыщенность: "
        f"{bright_low_sat_pct:.1f}%",
        fontsize=10
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

exposure_test_df = pd.DataFrame(results)

display(exposure_test_df)

In [ ]:
import cv2
import numpy as np
import pandas as pd

# Используем уже созданную таблицу
exposure_v2_df = exposure_df.copy()

bright_scores = []
low_sat_scores = []

for _, row in exposure_v2_df.iterrows():

    key = (
        row["class"].lower(),
        row["filename"]
    )

    paths = image_index.get(key, [])

    if not paths:
        bright_scores.append(np.nan)
        low_sat_scores.append(np.nan)
        continue

    image = cv2.imread(str(paths[0]))

    if image is None:
        bright_scores.append(np.nan)
        low_sat_scores.append(np.nan)
        continue

    # Переводим изображение в HSV
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    saturation = hsv[:, :, 1]
    brightness = hsv[:, :, 2]

    # Очень яркие пиксели
    bright_pct = (
        (brightness >= 245).mean() * 100
    )

    # Очень яркие и слабонасыщенные пиксели
    low_sat_pct = (
        (
            (brightness >= 245) &
            (saturation <= 100)
        ).mean() * 100
    )

    bright_scores.append(round(bright_pct, 2))
    low_sat_scores.append(round(low_sat_pct, 2))

exposure_v2_df["bright_v_pct"] = bright_scores
exposure_v2_df["bright_low_sat_pct"] = low_sat_scores

# Отмечаем уже известных кандидатов на исключение
known_bad = set(zip(
    quality_flags_df["class"],
    quality_flags_df["filename"]
))

exposure_v2_df["already_flagged"] = [
    (cls, filename) in known_bad
    for cls, filename in zip(
        exposure_v2_df["class"],
        exposure_v2_df["filename"]
    )
]

print(
    "Успешно проверено:",
    exposure_v2_df["bright_v_pct"].notna().sum()
)

# Статистика по всем шести классам
class_summary = (
    exposure_v2_df.groupby("class")
    .agg(
        total=("filename", "size"),
        median_bright=("bright_low_sat_pct", "median"),
        p95_bright=(
            "bright_low_sat_pct",
            lambda x: x.quantile(0.95)
        ),
        max_bright=("bright_low_sat_pct", "max")
    )
    .round(2)
)

print("\nРаспределение яркости по классам:")
display(class_summary)

# Находим изображения, которые ещё не отмечены
unflagged = exposure_v2_df[
    ~exposure_v2_df["already_flagged"]
].copy()

# По 3 изображения с максимальным показателем
# из каждого класса
top_new = (
    unflagged
    .sort_values(
        "bright_low_sat_pct",
        ascending=False
    )
    .groupby("class")
    .head(3)
    .sort_values(
        ["class", "bright_low_sat_pct"],
        ascending=[True, False]
    )
)

print("\n18 изображений для дополнительной проверки:")

display(
    top_new[
        [
            "class",
            "filename",
            "shooting",
            "white_pct",
            "bright_v_pct",
            "bright_low_sat_pct"
        ]
    ]
)

In [ ]:
import matplotlib.pyplot as plt
import cv2

# Сортируем по классам и степени подозрительности
exposure_review = top_new.sort_values(
    ["class", "bright_low_sat_pct"],
    ascending=[True, False]
).copy()

assert len(exposure_review) == 18

fig, axes = plt.subplots(6, 3, figsize=(15, 24))

for ax, (_, row) in zip(
    axes.flat,
    exposure_review.iterrows()
):
    key = (
        row["class"].lower(),
        row["filename"]
    )

    paths = image_index.get(key, [])

    image = cv2.imread(str(paths[0])) if paths else None

    if image is None:
        ax.set_title("Файл не найден / ошибка чтения")
        ax.axis("off")
        continue

    ax.imshow(
        cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    )

    ax.set_title(
        f"{row['class']}\n"
        f"{row['filename']}\n"
        f"HSV яркие: {row['bright_v_pct']:.1f}% | "
        f"Яркие + слабая насыщенность: "
        f"{row['bright_low_sat_pct']:.1f}%",
        fontsize=9
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 11 изображений, которые мы решили исключить
bad_images = {
    ("Capsicum", "20251117_134205_frame_135.jpg"),
    ("Capsicum", "20251117_134205_frame_136.jpg"),

    ("Carrot", "20251113_210803_frame_14.jpg"),
    ("Carrot", "20251113_210803_frame_15.jpg"),
    ("Carrot", "20251113_210803_frame_16.jpg"),

    ("Potato", "20251113_205825_frame_2.jpg"),
    ("Potato", "20251113_205825_frame_3.jpg"),
    ("Potato", "20251113_205825_frame_4.jpg"),

    ("Squash", "20251117_134731_frame_105.jpg"),
    ("Squash", "20251117_134731_frame_110.jpg"),
    ("Squash", "20251117_134731_frame_111.jpg")
}

# Создаём журнал текущей проверки
exposure_review_log = exposure_review.copy()

exposure_review_log["quality_decision"] = [
    "exclude_candidate"
    if (cls, filename) in bad_images
    else "keep_after_visual_review"
    for cls, filename in zip(
        exposure_review_log["class"],
        exposure_review_log["filename"]
    )
]

# Указываем причины
exposure_review_log["quality_reason"] = (
    "acceptable_after_visual_review"
)

bad_mask = (
    exposure_review_log["quality_decision"]
    == "exclude_candidate"
)

exposure_review_log.loc[
    bad_mask, "quality_reason"
] = "severe_overexposure"

exposure_review_log.loc[
    bad_mask &
    (exposure_review_log["class"] == "Capsicum"),
    "quality_reason"
] = "occlusion_and_cropping"

# Проверяем полноту решений
assert len(exposure_review_log) == 18
assert bad_mask.sum() == 11

# Добавляем только кандидатов на исключение
new_flags = exposure_review_log.loc[
    bad_mask,
    [
        "class", "filename", "shooting",
        "shooting_group", "white_pct",
        "quality_decision", "quality_reason"
    ]
].copy()

quality_flags_df = pd.concat(
    [quality_flags_df, new_flags],
    ignore_index=True
).drop_duplicates(
    subset=["class", "filename"],
    keep="last"
)

print("Всего кандидатов:", len(quality_flags_df))

print("\nПо классам:")
display(quality_flags_df.groupby("class").size())

print("\nРешения по 18 фотографиям:")
display(
    exposure_review_log[
        ["class", "filename", "quality_decision"]
    ]
)

In [ ]:
import matplotlib.pyplot as plt
import cv2
import math
import pandas as pd

# Выбираем видеосерию Squash
focus_df = phash_df[
    (phash_df["class"] == "Squash") &
    (phash_df["shooting"] == "20251117_134731")
].copy()

# Получаем номера кадров
focus_df["frame_num"] = pd.to_numeric(
    focus_df["filename"].str.extract(
        r"_frame_(\d+)\."
    )[0],
    errors="coerce"
)

# Проверяем кадры с 100 по 120
focus_df = focus_df[
    focus_df["frame_num"].between(100, 120)
].sort_values("frame_num")

# Уже отмеченные кандидаты на исключение
flagged_names = set(
    quality_flags_df[
        quality_flags_df["class"] == "Squash"
    ]["filename"]
)

print("Найдено кадров:", len(focus_df))
print("Номера:", focus_df["frame_num"].astype(int).tolist())

# Показываем последовательность
cols = 5
rows = math.ceil(len(focus_df) / cols)

fig, axes = plt.subplots(
    rows, cols,
    figsize=(18, rows * 3.8),
    squeeze=False
)

for ax in axes.flat:
    ax.axis("off")

for ax, (_, row) in zip(axes.flat, focus_df.iterrows()):

    paths = image_index.get(
        (row["class"].lower(), row["filename"]), []
    )

    image = cv2.imread(str(paths[0])) if paths else None

    if image is None:
        ax.set_title("Файл не найден")
        continue

    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

    status = (
        "УЖЕ ОТМЕЧЕН"
        if row["filename"] in flagged_names
        else "НЕ В СПИСКЕ"
    )

    ax.set_title(
        f"Кадр №{int(row['frame_num'])}\n{status}",
        fontsize=10
    )

plt.tight_layout()
plt.show()

In [ ]:
# Два дополнительных пересвеченных кадра
new_frame_numbers = [104, 112]

new_boundary_flags = focus_df[
    focus_df["frame_num"].isin(new_frame_numbers)
][
    ["class", "filename", "shooting", "shooting_group"]
].copy()

assert len(new_boundary_flags) == 2

# Добавляем предыдущие показатели экспозиции
new_boundary_flags = new_boundary_flags.merge(
    exposure_df[["class", "filename", "white_pct"]],
    on=["class", "filename"],
    how="left",
    validate="one_to_one"
)

new_boundary_flags["quality_decision"] = "exclude_candidate"
new_boundary_flags["quality_reason"] = "localized_overexposure"

# Обновляем общую таблицу
quality_flags_df = pd.concat(
    [quality_flags_df, new_boundary_flags],
    ignore_index=True
).drop_duplicates(
    subset=["class", "filename"],
    keep="last"
)

# Отдельно фиксируем пограничные случаи
squash_borderline_df = focus_df[
    focus_df["frame_num"].isin([100, 101])
][
    ["class", "filename", "shooting", "shooting_group"]
].copy()

squash_borderline_df["quality_decision"] = "manual_review"
squash_borderline_df["quality_reason"] = "possible_blur_or_cropping"

print("Всего кандидатов на исключение:", len(quality_flags_df))
print("Пограничных кадров:", len(squash_borderline_df))

print("\nКандидаты по классам:")
display(quality_flags_df.groupby("class").size())


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import math

# Выбираем нужную видеосерию Potato
potato_focus = phash_df[
    (phash_df["class"] == "Potato") &
    (phash_df["shooting"] == "20251113_205825")
].copy()

# Получаем номера кадров
potato_focus["frame_num"] = pd.to_numeric(
    potato_focus["filename"].str.extract(
        r"_frame_(\d+)\."
    )[0],
    errors="coerce"
)

# Показываем кадры 0–15
potato_focus = potato_focus[
    potato_focus["frame_num"].between(0, 15)
].sort_values("frame_num")

# Уже отмеченные изображения
flagged_names = set(
    quality_flags_df[
        quality_flags_df["class"] == "Potato"
    ]["filename"]
)

print("Найдено кадров:", len(potato_focus))
print(
    "Номера кадров:",
    potato_focus["frame_num"].astype(int).tolist()
)

# Визуализация
n = len(potato_focus)

if n > 0:
    cols = 4
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(
        rows, cols,
        figsize=(16, rows * 4),
        squeeze=False
    )

    for ax in axes.flat:
        ax.axis("off")

    for ax, (_, row) in zip(
        axes.flat, potato_focus.iterrows()
    ):
        paths = image_index.get(
            (row["class"].lower(), row["filename"]), []
        )

        image = cv2.imread(str(paths[0])) if paths else None

        if image is None:
            ax.set_title("Файл не найден")
            continue

        ax.imshow(
            cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        )

        status = (
            "УЖЕ ОТМЕЧЕН"
            if row["filename"] in flagged_names
            else "НЕ В СПИСКЕ"
        )

        ax.set_title(
            f"Кадр №{int(row['frame_num'])}\n{status}",
            fontsize=10
        )

    plt.tight_layout()
    plt.show()
else:
    print("Кадров в выбранном диапазоне нет.")

In [ ]:
# Новые кандидаты: кадры Potato 5–15
potato_new_bad = potato_focus[
    potato_focus["frame_num"].between(5, 15)
][
    ["class", "filename", "shooting", "shooting_group"]
].copy()

# Проверяем, что найдены все 11 кадров
assert len(potato_new_bad) == 11

# Добавляем ранее рассчитанные показатели экспозиции
potato_new_bad = potato_new_bad.merge(
    exposure_v2_df[
        ["class", "filename", "white_pct",
         "bright_low_sat_pct"]
    ],
    on=["class", "filename"],
    how="left",
    validate="one_to_one"
)

potato_new_bad["quality_decision"] = "exclude_candidate"
potato_new_bad["quality_reason"] = "severe_overexposure"

# Обновляем общую таблицу без повторов
quality_flags_df = pd.concat(
    [
        quality_flags_df,
        potato_new_bad[
            ["class", "filename", "shooting",
             "shooting_group", "white_pct",
             "quality_decision", "quality_reason"]
        ]
    ],
    ignore_index=True
).drop_duplicates(
    subset=["class", "filename"],
    keep="last"
)

print("Всего кандидатов:", len(quality_flags_df))

print("\nПо классам:")
display(quality_flags_df.groupby("class").size())

print("\nНовые кадры Potato и их показатели HSV:")
display(
    potato_new_bad[
        ["filename", "white_pct", "bright_low_sat_pct"]
    ].sort_values("filename")
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import math

# Выбираем всю нужную видеосерию
potato_next = phash_df[
    (phash_df["class"] == "Potato") &
    (phash_df["shooting"] == "20251113_205825")
].copy()

# Получаем номера кадров
potato_next["frame_num"] = pd.to_numeric(
    potato_next["filename"].str.extract(
        r"_frame_(\d+)\."
    )[0],
    errors="coerce"
)

# Выбираем кадры 16–35
potato_next = potato_next[
    potato_next["frame_num"].between(16, 35)
].sort_values("frame_num")

# Добавляем уже рассчитанные показатели HSV
potato_next = potato_next.merge(
    exposure_v2_df[
        ["class", "filename", "bright_low_sat_pct"]
    ],
    on=["class", "filename"],
    how="left",
    validate="one_to_one"
)

# Уже отмеченные кандидаты
flagged_names = set(
    quality_flags_df[
        quality_flags_df["class"] == "Potato"
    ]["filename"]
)

print("Найдено кадров:", len(potato_next))
print(
    "Номера:",
    potato_next["frame_num"].astype(int).tolist()
)

n = len(potato_next)

if n > 0:
    cols = 4
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(
        rows, cols,
        figsize=(16, rows * 4),
        squeeze=False
    )

    for ax in axes.flat:
        ax.axis("off")

    for ax, (_, row) in zip(
        axes.flat, potato_next.iterrows()
    ):
        paths = image_index.get(
            (row["class"].lower(), row["filename"]), []
        )

        image = cv2.imread(str(paths[0])) if paths else None

        if image is None:
            ax.set_title("Файл не найден")
            continue

        ax.imshow(
            cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        )

        status = (
            "УЖЕ ОТМЕЧЕН"
            if row["filename"] in flagged_names
            else "НЕ В СПИСКЕ"
        )

        ax.set_title(
            f"Кадр №{int(row['frame_num'])}\n"
            f"HSV: {row['bright_low_sat_pct']:.1f}%\n"
            f"{status}",
            fontsize=10
        )

    plt.tight_layout()
    plt.show()
else:
    print("Кадров 16–35 в этой видеосерии нет.")

In [ ]:
# Дополнительные пересвеченные кадры Potato
potato_extra = potato_next[
    potato_next["frame_num"].between(16, 19)
][
    ["class", "filename", "shooting", "shooting_group"]
].copy()

assert len(potato_extra) == 4

# Получаем ранее рассчитанную экспозицию
potato_extra = potato_extra.merge(
    exposure_v2_df[
        ["class", "filename", "white_pct"]
    ],
    on=["class", "filename"],
    how="left",
    validate="one_to_one"
)

potato_extra["quality_decision"] = "exclude_candidate"
potato_extra["quality_reason"] = "localized_overexposure"

# Обновляем таблицу кандидатов
quality_flags_df = pd.concat(
    [quality_flags_df, potato_extra],
    ignore_index=True
).drop_duplicates(
    subset=["class", "filename"],
    keep="last"
)

print("Всего кандидатов:", len(quality_flags_df))

print("\nПо классам:")
display(quality_flags_df.groupby("class").size())

print("\nНовые кандидаты Potato:")
display(potato_extra[["filename", "quality_reason"]])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import math

# Выбираем видеосерию Carrot
carrot_focus = phash_df[
    (phash_df["class"] == "Carrot") &
    (phash_df["shooting"] == "20251113_210803")
].copy()

# Получаем номера кадров
carrot_focus["frame_num"] = pd.to_numeric(
    carrot_focus["filename"].str.extract(
        r"_frame_(\d+)\."
    )[0],
    errors="coerce"
)

# Берём кадры 17–36
carrot_focus = carrot_focus[
    carrot_focus["frame_num"].between(17, 36)
].sort_values("frame_num")

# Добавляем рассчитанные показатели HSV
carrot_focus = carrot_focus.merge(
    exposure_v2_df[
        ["class", "filename", "bright_low_sat_pct"]
    ],
    on=["class", "filename"],
    how="left",
    validate="one_to_one"
)

# Проверяем, какие кадры уже отмечены
flagged_names = set(
    quality_flags_df[
        quality_flags_df["class"] == "Carrot"
    ]["filename"]
)

print("Найдено кадров:", len(carrot_focus))
print(
    "Номера кадров:",
    carrot_focus["frame_num"].astype(int).tolist()
)

# Отображаем фотографии
n = len(carrot_focus)

if n > 0:
    cols = 4
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(
        rows, cols,
        figsize=(16, rows * 4),
        squeeze=False
    )

    for ax in axes.flat:
        ax.axis("off")

    for ax, (_, row) in zip(
        axes.flat, carrot_focus.iterrows()
    ):

        paths = image_index.get(
            (row["class"].lower(), row["filename"]), []
        )

        image = cv2.imread(str(paths[0])) if paths else None

        if image is None:
            ax.set_title("Файл не найден")
            continue

        ax.imshow(
            cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        )

        status = (
            "УЖЕ ОТМЕЧЕН"
            if row["filename"] in flagged_names
            else "НЕ В СПИСКЕ"
        )

        ax.set_title(
            f"Кадр №{int(row['frame_num'])}\n"
            f"HSV: {row['bright_low_sat_pct']:.1f}%\n"
            f"{status}",
            fontsize=10
        )

    plt.tight_layout()
    plt.show()
else:
    print("В выбранном диапазоне кадров нет.")

In [ ]:
# Кадры Carrot 17–22: кандидаты на исключение
carrot_new_bad = carrot_focus[
    carrot_focus["frame_num"].between(17, 22)
][
    ["class", "filename", "shooting", "shooting_group"]
].copy()

assert len(carrot_new_bad) == 6

# Добавляем показатели предыдущей проверки
carrot_new_bad = carrot_new_bad.merge(
    exposure_df[
        ["class", "filename", "white_pct"]
    ],
    on=["class", "filename"],
    how="left",
    validate="one_to_one"
)

carrot_new_bad["quality_decision"] = "exclude_candidate"
carrot_new_bad["quality_reason"] = "localized_overexposure"

# Обновляем таблицу без повторов
quality_flags_df = pd.concat(
    [quality_flags_df, carrot_new_bad],
    ignore_index=True
).drop_duplicates(
    subset=["class", "filename"],
    keep="last"
)

# Кадры 23–26: пограничные случаи
carrot_borderline_df = carrot_focus[
    carrot_focus["frame_num"].between(23, 26)
][
    ["class", "filename", "shooting", "shooting_group"]
].copy()

carrot_borderline_df["quality_decision"] = "manual_review"
carrot_borderline_df["quality_reason"] = "moderate_overexposure"

print("Всего кандидатов:", len(quality_flags_df))
print("Пограничных кадров Carrot:", len(carrot_borderline_df))

display(quality_flags_df.groupby("class").size())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import cv2

# Собираем уже проверенные фотографии
reviewed_tables = [
    quality_flags_df,
    composition_review_log,
    exposure_review_log,
    squash_borderline_df,
    carrot_borderline_df
]

reviewed_keys = set()

for table in reviewed_tables:
    reviewed_keys.update(
        zip(table["class"], table["filename"])
    )

# Дополнительно исключаем ранее проверенный Capsicum
reviewed_keys.add(
    ("Capsicum", "20251113_205619.jpg")
)

# Формируем пул ещё не проверенных изображений
audit_pool = phash_df[
    ~phash_df.apply(
        lambda row: (
            row["class"], row["filename"]
        ) in reviewed_keys,
        axis=1
    )
].copy()

# Выбираем по 4 фотографии каждого класса,
# по возможности из разных групп съёмки
samples = []

for cls, group in audit_pool.groupby("class"):

    representatives = (
        group.groupby(
            "shooting_group",
            dropna=False,
            group_keys=False
        )
        .sample(n=1, random_state=42)
    )

    if len(representatives) > 4:
        representatives = representatives.sample(
            n=4, random_state=42
        )

    remaining = group.drop(
        index=representatives.index
    )

    need = 4 - len(representatives)

    extra = remaining.sample(
        n=min(need, len(remaining)),
        random_state=42
    )

    samples.append(
        pd.concat([representatives, extra])
    )

audit_sample = pd.concat(
    samples, ignore_index=True
).sort_values(
    ["class", "shooting_group"]
).reset_index(drop=True)

print("Контрольных фотографий:", len(audit_sample))

print("\nРаспределение по классам:")
print(audit_sample["class"].value_counts())

# Показываем фотографии
fig, axes = plt.subplots(6, 4, figsize=(16, 25))

for ax in axes.flat:
    ax.axis("off")

for ax, (_, row) in zip(
    axes.flat, audit_sample.iterrows()
):
    paths = image_index.get(
        (row["class"].lower(), row["filename"]), []
    )

    image = cv2.imread(str(paths[0])) if paths else None

    if image is None:
        ax.set_title("Ошибка чтения")
        continue

    ax.imshow(
        cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    )

    ax.set_title(
        f"{row['class']}\n"
        f"{row['filename']}\n"
        f"{row['shooting_group']}",
        fontsize=8
    )

plt.tight_layout()
plt.show()

In [ ]:
# Проверяем соседей подозрительного кадра 159
squash_focus_159 = phash_df[
    (phash_df["class"] == "Squash") &
    (phash_df["shooting"] == "20251117_134731")
].copy()

# Номер кадра
squash_focus_159["frame_num"] = pd.to_numeric(
    squash_focus_159["filename"].str.extract(
        r"_frame_(\d+)\."
    )[0],
    errors="coerce"
)

# Кадры 152–166
squash_focus_159 = squash_focus_159[
    squash_focus_159["frame_num"].between(152, 166)
].sort_values("frame_num")

# Добавляем показатели HSV
squash_focus_159 = squash_focus_159.merge(
    exposure_v2_df[
        ["class", "filename", "bright_low_sat_pct"]
    ],
    on=["class", "filename"],
    how="left",
    validate="one_to_one"
)

# Проверяем уже отмеченные фотографии
flagged_names = set(
    quality_flags_df[
        quality_flags_df["class"] == "Squash"
    ]["filename"]
)

print("Найдено кадров:", len(squash_focus_159))

fig, axes = plt.subplots(4, 4, figsize=(16, 16))

for ax in axes.flat:
    ax.axis("off")

for ax, (_, row) in zip(
    axes.flat, squash_focus_159.iterrows()
):
    paths = image_index.get(
        (row["class"].lower(), row["filename"]), []
    )

    image = cv2.imread(str(paths[0])) if paths else None

    if image is None:
        ax.set_title("Ошибка чтения")
        continue

    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

    status = (
        "УЖЕ ОТМЕЧЕН"
        if row["filename"] in flagged_names
        else "НЕ В СПИСКЕ"
    )

    ax.set_title(
        f"Кадр №{int(row['frame_num'])}\n"
        f"HSV: {row['bright_low_sat_pct']:.1f}%\n"
        f"{status}",
        fontsize=10
    )

plt.tight_layout()
plt.show()

In [ ]:
# Отмечаем кадры Squash 155–166
new_bad = squash_focus_159[
    squash_focus_159["frame_num"].between(155, 166)
][
    ["class", "filename", "shooting", "shooting_group"]
].copy()

assert len(new_bad) == 12

# Добавляем показатели экспозиции
new_bad = new_bad.merge(
    exposure_df[
        ["class", "filename", "white_pct"]
    ],
    on=["class", "filename"],
    how="left",
    validate="one_to_one"
)

new_bad["quality_decision"] = "exclude_candidate"
new_bad["quality_reason"] = "localized_overexposure"

# Обновляем общую таблицу
quality_flags_df = pd.concat(
    [quality_flags_df, new_bad],
    ignore_index=True
).drop_duplicates(
    subset=["class", "filename"],
    keep="last"
)

print("Всего кандидатов:", len(quality_flags_df))

print("\nПо классам:")
display(quality_flags_df.groupby("class").size())

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

from tqdm.auto import tqdm

# ==================================================
# 1. Фиксируем последние 12 кадров Squash, если нужно
# ==================================================

squash_names = [
    f"20251117_134731_frame_{i}.jpg"
    for i in range(155, 167)
]

last_flags = phash_df[
    (phash_df["class"] == "Squash") &
    phash_df["filename"].isin(squash_names)
][
    ["class", "filename", "shooting", "shooting_group"]
].copy()

assert len(last_flags) == 12

last_flags = last_flags.merge(
    exposure_df[["class", "filename", "white_pct"]],
    on=["class", "filename"],
    how="left",
    validate="one_to_one"
)

last_flags["quality_decision"] = "exclude_candidate"
last_flags["quality_reason"] = "localized_overexposure"

quality_flags_df = pd.concat(
    [quality_flags_df, last_flags],
    ignore_index=True
).drop_duplicates(
    subset=["class", "filename"],
    keep="first"
).reset_index(drop=True)

print("Кандидатов на исключение:", len(quality_flags_df))


# ==================================================
# 2. Отделяем кадры видеосерий от одиночных фотографий
# ==================================================

frame_numbers = phash_df["filename"].str.extract(
    r"_frame_(\d+)\.(?:jpg|jpeg|png)$",
    flags=re.IGNORECASE
)[0]

video_quality_df = phash_df[
    frame_numbers.notna()
].copy()

video_quality_df["frame_num"] = (
    frame_numbers.loc[video_quality_df.index]
    .astype(int)
)

print("Кадров видеосерий:", len(video_quality_df))
print(
    "Количество видеосерий:",
    video_quality_df.groupby(["class", "shooting"]).ngroups
)


# ==================================================
# 3. Добавляем уже рассчитанные показатели экспозиции
# ==================================================

video_quality_df = video_quality_df.merge(
    exposure_v2_df[
        ["class", "filename", "bright_low_sat_pct"]
    ],
    on=["class", "filename"],
    how="left",
    validate="one_to_one"
)


# ==================================================
# 4. Рассчитываем резкость для кадров видеосерий
# ==================================================

def video_sharpness(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    h, w = gray.shape
    new_h = max(1, round(h * 256 / w))

    gray = cv2.resize(
        gray,
        (256, new_h),
        interpolation=cv2.INTER_AREA
    )

    return cv2.Laplacian(
        gray, cv2.CV_64F
    ).var()


sharpness_values = []

for row in tqdm(
    video_quality_df.itertuples(index=False),
    total=len(video_quality_df),
    desc="Анализ резкости"
):
    paths = image_index.get(
        (row[0].lower(), row[1]), []
    )

    image = cv2.imread(str(paths[0])) if paths else None

    if image is None:
        sharpness_values.append(np.nan)
    else:
        sharpness_values.append(
            video_sharpness(image)
        )

video_quality_df["sharpness"] = sharpness_values


# ==================================================
# 5. Отмечаем уже найденные плохие фотографии
# ==================================================

flagged_keys = set(zip(
    quality_flags_df["class"],
    quality_flags_df["filename"]
))

video_quality_df["flagged"] = [
    (cls, filename) in flagged_keys
    for cls, filename in zip(
        video_quality_df["class"],
        video_quality_df["filename"]
    )
]


# ==================================================
# 6. Сводная статистика по всем видеосериям
# ==================================================

video_summary = (
    video_quality_df
    .groupby(["class", "shooting"])
    .agg(
        frames=("filename", "size"),
        flagged=("flagged", "sum"),
        hsv_p95=(
            "bright_low_sat_pct",
            lambda x: x.quantile(0.95)
        ),
        bright_15=(
            "bright_low_sat_pct",
            lambda x: (x >= 15).sum()
        ),
        median_sharpness=("sharpness", "median")
    )
    .reset_index()
)

video_summary["bright_15_pct"] = (
    video_summary["bright_15"]
    / video_summary["frames"]
    * 100
)

print("\nОбщая статистика видеосерий:")

display(
    video_summary.sort_values(
        ["flagged", "bright_15_pct"],
        ascending=False
    ).round(2)
)


# ==================================================
# 7. Строим общую карту качества
# ==================================================

groups = list(
    video_quality_df.groupby(["class", "shooting"])
)

n_series = len(groups)
n_bins = 40

brightness_map = np.full(
    (n_series, n_bins), np.nan
)

sharpness_map = np.full(
    (n_series, n_bins), np.nan
)

flag_positions = []
labels = []

for row_idx, ((cls, shooting), group) in enumerate(groups):

    group = group.sort_values("frame_num").copy()

    labels.append(
        f"{cls} | {shooting} ({len(group)})"
    )

    first = group["frame_num"].min()
    last = group["frame_num"].max()

    if first == last:
        group["bin"] = 0
    else:
        group["bin"] = (
            (group["frame_num"] - first)
            / (last - first)
            * (n_bins - 1)
        ).astype(int)

    for bin_idx, part in group.groupby("bin"):

        brightness_map[row_idx, bin_idx] = (
            part["bright_low_sat_pct"].max()
        )

        sharpness_map[row_idx, bin_idx] = (
            part["sharpness"].median()
        )

    for bin_idx in group.loc[group["flagged"], "bin"].unique():
        flag_positions.append((row_idx, bin_idx))


# Логарифмическая шкала для удобства отображения резкости
sharpness_display = np.log1p(sharpness_map)

valid_sharpness = sharpness_display[
    np.isfinite(sharpness_display)
]

sharp_min, sharp_max = np.percentile(
    valid_sharpness, [5, 95]
)

fig, axes = plt.subplots(
    1, 2,
    figsize=(21, max(10, n_series * 0.65)),
    sharey=True
)

im1 = axes[0].imshow(
    brightness_map,
    aspect="auto",
    interpolation="nearest",
    cmap="magma",
    vmin=0,
    vmax=40
)

im2 = axes[1].imshow(
    sharpness_display,
    aspect="auto",
    interpolation="nearest",
    cmap="viridis",
    vmin=sharp_min,
    vmax=sharp_max
)

axes[0].set_title(
    "Пересвет: очень яркие слабонасыщенные пиксели",
    fontsize=12
)

axes[1].set_title(
    "Резкость: показатель Лапласиана",
    fontsize=12
)

for ax in axes:
    ax.set_yticks(range(n_series))
    ax.set_yticklabels(labels, fontsize=8)

    ax.set_xticks([0, 10, 20, 29, 39])
    ax.set_xticklabels(["0%", "25%", "50%", "75%", "100%"])

    ax.set_xlabel("Положение внутри видеосерии")

    for row_idx, bin_idx in flag_positions:
        ax.scatter(
            bin_idx, row_idx,
            color="cyan",
            marker="x",
            s=15,
            linewidths=0.8
        )

fig.colorbar(
    im1,
    ax=axes[0],
    fraction=0.025,
    pad=0.02,
    label="Доля ярких пикселей, %"
)

fig.colorbar(
    im2,
    ax=axes[1],
    fraction=0.025,
    pad=0.02,
    label="log(1 + резкость)"
)

plt.tight_layout()
plt.show()

print("\nКарта качества построена.")
print("Видеосерий на карте:", n_series)
print(
    "Кадров с рассчитанной резкостью:",
    video_quality_df["sharpness"].notna().sum()
)
print(
    "Уже отмеченных кадров на карте:",
    video_quality_df["flagged"].sum()
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import math

# 1. Берём уже рассчитанные показатели
# Ничего повторно не вычисляем

bright_candidates = video_quality_df[
    (video_quality_df["bright_low_sat_pct"] >= 15) &
    (~video_quality_df["flagged"])
].copy()

bright_candidates = bright_candidates.sort_values(
    ["class", "shooting", "frame_num"]
)

print(
    "Ещё не отмеченных кадров с HSV >= 15%:",
    len(bright_candidates)
)

# 2. Объединяем соседние кадры в участки
bright_candidates["segment"] = (
    bright_candidates
    .groupby(["class", "shooting"])["frame_num"]
    .transform(
        lambda x: x.diff().ne(1).cumsum()
    )
)

# 3. Сводная таблица всех подозрительных участков
bright_segments_df = (
    bright_candidates
    .groupby(["class", "shooting", "segment"])
    .agg(
        first_frame=("frame_num", "min"),
        last_frame=("frame_num", "max"),
        frames=("filename", "size"),
        max_hsv=("bright_low_sat_pct", "max")
    )
    .reset_index()
    .sort_values(
        ["frames", "max_hsv"],
        ascending=False
    )
)

print("\nПодозрительные участки по всем видеосериям:")
display(bright_segments_df)

# 4. Находим наиболее яркий кадр каждого участка
keys = ["class", "shooting", "segment"]

peak_indices = (
    bright_candidates.groupby(keys)[
        "bright_low_sat_pct"
    ].idxmax()
)

bright_examples = (
    bright_candidates.loc[peak_indices]
    .merge(
        bright_segments_df,
        on=keys,
        how="left",
        validate="one_to_one"
    )
    .sort_values(
        ["frames", "max_hsv"],
        ascending=False
    )
)

print("\nВсего найдено участков:", len(bright_segments_df))

# 5. Визуализируем до 32 приоритетных участков
examples_to_show = bright_examples.head(32)

n = len(examples_to_show)

if n > 0:
    cols = 4
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(
        rows, cols,
        figsize=(17, rows * 4),
        squeeze=False
    )

    for ax in axes.flat:
        ax.axis("off")

    for ax, (_, row) in zip(
        axes.flat,
        examples_to_show.iterrows()
    ):
        paths = image_index.get(
            (row["class"].lower(), row["filename"]), []
        )

        image = (
            cv2.imread(str(paths[0]))
            if paths else None
        )

        if image is None:
            ax.set_title("Файл не найден")
            continue

        ax.imshow(
            cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        )

        ax.set_title(
            f"{row['class']} | {row['shooting']}\n"
            f"Участок {int(row['first_frame'])}–"
            f"{int(row['last_frame'])} "
            f"({int(row['frames'])} кадров)\n"
            f"Кадр {int(row['frame_num'])} | "
            f"HSV: {row['bright_low_sat_pct']:.1f}%",
            fontsize=9
        )

    plt.tight_layout()
    plt.show()
else:
    print("Новых подозрительных участков не найдено.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import math

# 1. Собираем уже подтверждённые хорошие фотографии
reviewed_good = pd.concat([
    composition_review_log,
    exposure_review_log
], ignore_index=True)

reviewed_good = reviewed_good[
    reviewed_good["quality_decision"] ==
    "keep_after_visual_review"
]

good_keys = set(zip(
    reviewed_good["class"],
    reviewed_good["filename"]
))

# 2. На всякий случай исключаем уже отмеченные плохие
bad_keys = set(zip(
    quality_flags_df["class"],
    quality_flags_df["filename"]
))

# 3. Берём все подозрительные кадры
full_bright_review = bright_candidates.copy()

full_bright_review = full_bright_review[
    ~full_bright_review.apply(
        lambda r: (
            r["class"], r["filename"]
        ) in (good_keys | bad_keys),
        axis=1
    )
].copy()

full_bright_review = full_bright_review.sort_values(
    ["class", "shooting", "frame_num"]
)

print(
    "Фотографий для ручной проверки:",
    len(full_bright_review)
)

print("\nПо видеосериям:")
display(
    full_bright_review.groupby(
        ["class", "shooting"]
    ).size().reset_index(name="count")
)

# 4. Показываем все фотографии одним контактным листом
n = len(full_bright_review)

if n > 0:

    cols = 4
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(
        rows, cols,
        figsize=(17, rows * 4),
        squeeze=False
    )

    for ax in axes.flat:
        ax.axis("off")

    for ax, (_, row) in zip(
        axes.flat,
        full_bright_review.iterrows()
    ):

        paths = image_index.get(
            (row["class"].lower(), row["filename"]), []
        )

        image = cv2.imread(
            str(paths[0])
        ) if paths else None

        if image is None:
            ax.set_title("Файл не найден")
            continue

        ax.imshow(
            cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        )

        ax.set_title(
            f"{row['class']} | {row['shooting']}\n"
            f"Кадр №{int(row['frame_num'])}\n"
            f"HSV: {row['bright_low_sat_pct']:.1f}%",
            fontsize=10
        )

    plt.tight_layout()
    plt.show()

else:
    print("Непроверенных фотографий нет.")

In [ ]:
# Выбираем проблемные участки
bad_mask = (
    (
        (full_bright_review["class"] == "Squash") &
        (full_bright_review["shooting"] == "20251113_210502") &
        (full_bright_review["frame_num"].isin(
            [0, 3, 4, 5] + list(range(7, 20))
        ))
    )
    |
    (
        (full_bright_review["class"] == "Carrot") &
        (full_bright_review["shooting"] == "20251113_210822") &
        (full_bright_review["frame_num"].between(1, 5))
    )
    |
    (
        (full_bright_review["class"] == "Squash") &
        (full_bright_review["shooting"] == "20251117_134731") &
        (full_bright_review["frame_num"].between(47, 51))
    )
)

assert bad_mask.sum() == 27, "Проверь состав отобранных кадров"

# Новые кандидаты
new_bad = full_bright_review.loc[
    bad_mask,
    ["class", "filename", "shooting", "shooting_group"]
].copy()

new_bad = new_bad.merge(
    exposure_df[["class", "filename", "white_pct"]],
    on=["class", "filename"],
    how="left",
    validate="one_to_one"
)

new_bad["quality_decision"] = "exclude_candidate"
new_bad["quality_reason"] = "visual_overexposure"

# Обновляем общую таблицу
quality_flags_df = pd.concat(
    [quality_flags_df, new_bad],
    ignore_index=True
).drop_duplicates(
    subset=["class", "filename"],
    keep="first"
).reset_index(drop=True)

# Оставшиеся 6 — пограничные
bright_manual_review_df = full_bright_review.loc[
    ~bad_mask,
    ["class", "filename", "shooting",
     "shooting_group", "frame_num"]
].copy()

bright_manual_review_df["quality_decision"] = "manual_review"
bright_manual_review_df["quality_reason"] = "borderline_quality"

print("Всего кандидатов:", len(quality_flags_df))
print("Пограничных в текущей проверке:", len(bright_manual_review_df))

print("\nРаспределение по классам:")
display(quality_flags_df.groupby("class").size())

print("\nПограничные фотографии:")
display(bright_manual_review_df)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2

# =========================================
# 1. Исключаем уже отмеченные плохие кадры
# =========================================

bad_keys = set(zip(
    quality_flags_df["class"],
    quality_flags_df["filename"]
))

blur_pool = video_quality_df.copy()

blur_pool["already_flagged"] = [
    (cls, filename) in bad_keys
    for cls, filename in zip(
        blur_pool["class"],
        blur_pool["filename"]
    )
]

blur_pool = blur_pool[
    (~blur_pool["already_flagged"]) &
    (blur_pool["sharpness"].notna())
].copy()

print("Кадров для анализа:", len(blur_pool))


# =========================================
# 2. Статистика резкости по видеосериям
# =========================================

blur_summary_df = (
    blur_pool
    .groupby(["class", "shooting"])
    .agg(
        frames=("filename", "size"),
        min_sharpness=("sharpness", "min"),
        p10_sharpness=(
            "sharpness",
            lambda x: x.quantile(0.10)
        ),
        median_sharpness=("sharpness", "median")
    )
    .reset_index()
)

# Относительная резкость нижних 10% кадров
blur_summary_df["p10_to_median"] = (
    blur_summary_df["p10_sharpness"] /
    blur_summary_df["median_sharpness"].replace(0, np.nan)
)

print("\nСтатистика всех видеосерий:")

display(
    blur_summary_df.sort_values(
        "p10_to_median"
    ).round(2)
)


# =========================================
# 3. Выбираем три разных кадра
#    из каждой видеосерии
# =========================================

samples = []

sample_levels = [
    ("Минимальная", 0.00),
    ("Нижние 10%", 0.10),
    ("Типичная", 0.50)
]

for (cls, shooting), group in blur_pool.groupby(
    ["class", "shooting"]
):

    used_indices = set()

    for label, quantile in sample_levels:

        available = group.loc[
            ~group.index.isin(used_indices)
        ]

        if available.empty:
            continue

        target = group["sharpness"].quantile(quantile)

        idx = (
            (available["sharpness"] - target)
            .abs()
            .idxmin()
        )

        used_indices.add(idx)

        row = group.loc[idx]

        samples.append({
            "class": cls,
            "shooting": shooting,
            "filename": row["filename"],
            "frame_num": int(row["frame_num"]),
            "sharpness": row["sharpness"],
            "sample_type": label
        })

blur_audit_df = pd.DataFrame(samples)

print("\nВыбрано кадров:", len(blur_audit_df))


# =========================================
# 4. Показываем контактные листы
#    по всем 19 видеосериям
# =========================================

series_keys = list(
    blur_audit_df.groupby(
        ["class", "shooting"],
        sort=True
    ).groups.keys()
)

# Делим вывод на страницы по 7 видеосерий
# для более удобного просмотра

for start in range(0, len(series_keys), 7):

    page_keys = series_keys[start:start + 7]

    fig, axes = plt.subplots(
        len(page_keys),
        3,
        figsize=(13, len(page_keys) * 3.2),
        squeeze=False
    )

    for ax in axes.flat:
        ax.axis("off")

    for i, (cls, shooting) in enumerate(page_keys):

        part = blur_audit_df[
            (blur_audit_df["class"] == cls) &
            (blur_audit_df["shooting"] == shooting)
        ]

        for j, label in enumerate(
            ["Минимальная", "Нижние 10%", "Типичная"]
        ):

            selected = part[
                part["sample_type"] == label
            ]

            if selected.empty:
                continue

            row = selected.iloc[0]

            paths = image_index.get(
                (cls.lower(), row["filename"]), []
            )

            image = (
                cv2.imread(str(paths[0]))
                if paths else None
            )

            if image is None:
                axes[i, j].set_title("Файл не найден")
                continue

            axes[i, j].imshow(
                cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            )

            axes[i, j].set_title(
                f"{cls} | {shooting}\n"
                f"{label} | Кадр №{row['frame_num']}\n"
                f"Резкость: {row['sharpness']:.1f}",
                fontsize=9
            )

            axes[i, j].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import math

# ==============================================
# 1. Берём готовые показатели резкости
# ==============================================

bad_keys = set(zip(
    quality_flags_df["class"],
    quality_flags_df["filename"]
))

blur_scan_df = video_quality_df.copy()

blur_scan_df = blur_scan_df[
    ~blur_scan_df.apply(
        lambda r: (r["class"], r["filename"]) in bad_keys,
        axis=1
    )
].copy()

blur_scan_df = blur_scan_df.dropna(
    subset=["sharpness", "frame_num"]
)

# ==============================================
# 2. Сравниваем резкость внутри каждой видеосерии
# ==============================================

group_cols = ["class", "shooting"]

stats = (
    blur_scan_df.groupby(group_cols)["sharpness"]
    .agg(
        median_sharpness="median",
        p20_sharpness=lambda x: x.quantile(0.20)
    )
    .reset_index()
)

blur_scan_df = blur_scan_df.merge(
    stats,
    on=group_cols,
    how="left",
    validate="many_to_one"
)

blur_scan_df["sharpness_ratio"] = (
    blur_scan_df["sharpness"] /
    blur_scan_df["median_sharpness"].replace(0, np.nan)
)

# Два условия для подозрительных кадров
blur_scan_df["blur_suspect"] = (
    (blur_scan_df["sharpness"] <=
     blur_scan_df["p20_sharpness"]) &
    (blur_scan_df["sharpness_ratio"] < 0.45)
)

# ==============================================
# 3. Находим соседние подозрительные кадры
# ==============================================

blur_suspects_df = blur_scan_df[
    blur_scan_df["blur_suspect"]
].copy()

blur_suspects_df = blur_suspects_df.sort_values(
    ["class", "shooting", "frame_num"]
)

blur_suspects_df["segment"] = (
    blur_suspects_df
    .groupby(group_cols)["frame_num"]
    .transform(lambda x: x.diff().ne(1).cumsum())
)

segment_cols = ["class", "shooting", "segment"]

blur_segments_df = (
    blur_suspects_df
    .groupby(segment_cols)
    .agg(
        first_frame=("frame_num", "min"),
        last_frame=("frame_num", "max"),
        frames=("filename", "size"),
        min_sharpness=("sharpness", "min"),
        min_ratio=("sharpness_ratio", "min")
    )
    .reset_index()
    .sort_values(
        ["frames", "min_ratio"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

print("Кадров с подозрительной резкостью:",
      len(blur_suspects_df))

print("Подозрительных участков:",
      len(blur_segments_df))

print("\nВсе найденные участки:")
display(blur_segments_df.round(2))

# ==============================================
# 4. Выбираем худший кадр каждого участка
# ==============================================

if len(blur_segments_df) > 0:

    peak_idx = (
        blur_suspects_df.groupby(segment_cols)["sharpness"]
        .idxmin()
    )

    worst_examples = blur_suspects_df.loc[
        peak_idx
    ].copy()

    worst_examples = worst_examples.merge(
        blur_segments_df[
            segment_cols + [
                "first_frame", "last_frame", "frames"
            ]
        ],
        on=segment_cols,
        how="left",
        validate="one_to_one"
    )

    worst_examples = worst_examples.sort_values(
        ["frames", "sharpness_ratio"],
        ascending=[False, True]
    ).head(24)

    # ==============================================
    # 5. Показываем худший и типичный кадры
    # ==============================================

    def show_image(ax, row, title):

        paths = image_index.get(
            (row["class"].lower(), row["filename"]), []
        )

        image = cv2.imread(
            str(paths[0])
        ) if paths else None

        if image is not None:
            ax.imshow(
                cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            )

        ax.set_title(title, fontsize=10)
        ax.axis("off")

    # До 8 участков на одной странице
    for start in range(0, len(worst_examples), 8):

        page = worst_examples.iloc[start:start + 8]

        fig, axes = plt.subplots(
            len(page), 2,
            figsize=(11, len(page) * 3.2),
            squeeze=False
        )

        for i, (_, worst) in enumerate(page.iterrows()):

            group = blur_scan_df[
                (blur_scan_df["class"] == worst["class"]) &
                (blur_scan_df["shooting"] == worst["shooting"])
            ]

            # Кадр с резкостью, близкой к медиане
            target = group["sharpness"].median()

            typical_idx = (
                (group["sharpness"] - target)
                .abs()
                .idxmin()
            )

            typical = group.loc[typical_idx]

            show_image(
                axes[i, 0],
                worst,
                f"{worst['class']} | {worst['shooting']}\n"
                f"Участок {int(worst['first_frame'])}–"
                f"{int(worst['last_frame'])} "
                f"({int(worst['frames'])} кадров)\n"
                f"Подозрительный №{int(worst['frame_num'])}"
                f" | Резкость {worst['sharpness']:.1f}"
            )

            show_image(
                axes[i, 1],
                typical,
                f"Типичный кадр той же видеосерии\n"
                f"Кадр №{int(typical['frame_num'])}\n"
                f"Резкость {typical['sharpness']:.1f}"
            )

        plt.tight_layout()
        plt.show()

else:
    print("Участков по выбранным условиям не обнаружено.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

# 12 самых длинных подозрительных участков
top_blur_segments = (
    blur_segments_df
    .sort_values("frames", ascending=False)
    .head(12)
    .reset_index(drop=True)
)

print("Выбрано участков:", len(top_blur_segments))
print(
    "Подозрительных кадров внутри них:",
    top_blur_segments["frames"].sum()
)

display(
    top_blur_segments[
        ["class", "shooting",
         "first_frame", "last_frame", "frames"]
    ]
)

# Показываем по 6 участков на одной странице
for page_start in range(0, len(top_blur_segments), 6):

    page = top_blur_segments.iloc[
        page_start:page_start + 6
    ]

    fig, axes = plt.subplots(
        len(page), 6,
        figsize=(21, len(page) * 3.5),
        squeeze=False
    )

    for i, (_, segment) in enumerate(page.iterrows()):

        part = blur_suspects_df[
            (blur_suspects_df["class"] == segment["class"]) &
            (blur_suspects_df["shooting"] == segment["shooting"]) &
            (blur_suspects_df["segment"] == segment["segment"])
        ].sort_values("frame_num").reset_index(drop=True)

        n = len(part)

        # Пять точек по всей длине участка
        positions = [
            0,
            round((n - 1) * 0.25),
            round((n - 1) * 0.50),
            round((n - 1) * 0.75),
            n - 1
        ]

        # Кадр с минимальным показателем резкости
        worst_pos = part["sharpness"].idxmin()

        selected = [
            ("Начало", positions[0]),
            ("25%", positions[1]),
            ("Худший", worst_pos),
            ("50%", positions[2]),
            ("75%", positions[3]),
            ("Конец", positions[4])
        ]

        for j, (label, pos) in enumerate(selected):

            row = part.iloc[pos]
            ax = axes[i, j]

            paths = image_index.get(
                (row["class"].lower(), row["filename"]), []
            )

            image = (
                cv2.imread(str(paths[0]))
                if paths else None
            )

            if image is not None:
                ax.imshow(
                    cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                )

            ax.set_title(
                f"{segment['class']} | {segment['shooting']}\n"
                f"{label}: кадр {int(row['frame_num'])}\n"
                f"Резкость: {row['sharpness']:.1f}",
                fontsize=8
            )

            ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2

# 1. Берём фотографии, не отмеченные на исключение
bad_keys = set(zip(
    quality_flags_df["class"],
    quality_flags_df["filename"]
))

redundancy_df = phash_df.copy()

redundancy_df = redundancy_df[
    ~pd.MultiIndex.from_frame(
        redundancy_df[["class", "filename"]]
    ).isin(bad_keys)
].copy()

# 2. Оставляем кадры видеосерий
redundancy_df["frame_num"] = pd.to_numeric(
    redundancy_df["filename"].str.extract(
        r"_frame_(\d+)\."
    )[0],
    errors="coerce"
)

redundancy_df = redundancy_df.dropna(
    subset=["frame_num", "phash"]
).copy()

redundancy_df["frame_num"] = (
    redundancy_df["frame_num"].astype(int)
)

redundancy_df = redundancy_df.sort_values(
    ["class", "shooting", "frame_num"]
)

# 3. Находим предыдущий кадр в каждой видеосерии
groups = redundancy_df.groupby(["class", "shooting"])

redundancy_df["prev_filename"] = groups["filename"].shift()
redundancy_df["prev_frame"] = groups["frame_num"].shift()
redundancy_df["prev_phash"] = groups["phash"].shift()

# Сравниваем только действительно соседние кадры
phash_pairs_df = redundancy_df[
    (redundancy_df["frame_num"] -
     redundancy_df["prev_frame"]) == 1
].copy()

# 4. Расстояние Хэмминга между pHash
phash_pairs_df["distance"] = [
    (
        int(str(a), 16) ^ int(str(b), 16)
    ).bit_count()
    for a, b in zip(
        phash_pairs_df["phash"],
        phash_pairs_df["prev_phash"]
    )
]

# 5. Статистика по всем видеосериям
redundancy_summary_df = (
    phash_pairs_df
    .groupby(["class", "shooting"])
    .agg(
        pairs=("distance", "size"),
        median_distance=("distance", "median"),
        close_4=("distance", lambda x: (x <= 4).sum()),
        close_8=("distance", lambda x: (x <= 8).sum())
    )
    .reset_index()
)

redundancy_summary_df["close_4_pct"] = (
    100 * redundancy_summary_df["close_4"]
    / redundancy_summary_df["pairs"]
)

redundancy_summary_df["close_8_pct"] = (
    100 * redundancy_summary_df["close_8"]
    / redundancy_summary_df["pairs"]
)

print("Проверено соседних пар:", len(phash_pairs_df))

print("\nПовторяемость по видеосериям:")
display(
    redundancy_summary_df.sort_values(
        "close_4_pct", ascending=False
    ).round(1)
)

# 6. Выбираем примеры при разном расстоянии pHash
phash_pairs_df["similarity_band"] = pd.cut(
    phash_pairs_df["distance"],
    bins=[-1, 2, 4, 8, 64],
    labels=["0–2", "3–4", "5–8", "9–64"]
)

examples = []

for band, part in phash_pairs_df.groupby(
    "similarity_band", observed=True
):
    # По возможности берём разные видеосерии
    representatives = (
        part.groupby(["class", "shooting"], group_keys=False)
        .sample(n=1, random_state=42)
    )

    selected = representatives.sample(
        n=min(3, len(representatives)),
        random_state=42
    )

    examples.append(selected)

phash_examples_df = pd.concat(
    examples, ignore_index=True
).sort_values("distance")

print("\nПар для визуальной проверки:",
      len(phash_examples_df))

# 7. Показываем пары изображений
for start in range(0, len(phash_examples_df), 6):

    page = phash_examples_df.iloc[start:start + 6]

    fig, axes = plt.subplots(
        len(page), 2,
        figsize=(11, len(page) * 3.3),
        squeeze=False
    )

    for i, (_, row) in enumerate(page.iterrows()):

        for j, filename in enumerate([
            row["prev_filename"],
            row["filename"]
        ]):

            ax = axes[i, j]

            paths = image_index.get(
                (row["class"].lower(), filename), []
            )

            img = (
                cv2.imread(str(paths[0]))
                if paths else None
            )

            if img is not None:
                ax.imshow(
                    cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                )

            ax.set_title(
                f"{row['class']} | {row['shooting']}\n"
                f"{filename}\n"
                f"pHash расстояние: {row['distance']}",
                fontsize=9
            )

            ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:

# 1. Распределение pHash-расстояний

bins = [-1, 4, 8, 16, 24, 32, 64]
labels = ["0–4", "5–8", "9–16", "17–24", "25–32", "33–64"]

distance_ranges = pd.cut(
    phash_pairs_df["distance"],
    bins=bins,
    labels=labels
)

distribution = (
    distance_ranges.value_counts(sort=False)
    .rename_axis("distance_range")
    .reset_index(name="pairs")
)

distribution["percent"] = (
    100 * distribution["pairs"] / len(phash_pairs_df)
)

print("Распределение расстояний:")
display(distribution.round(1))

print("\nПроцентили:")
print(
    phash_pairs_df["distance"]
    .quantile([0.5, 0.75, 0.9, 0.95, 0.99])
)

# 2. Выбираем примеры с большими расстояниями

selected_parts = []

for low, high in [(17, 24), (25, 32), (33, 64)]:

    part = phash_pairs_df[
        phash_pairs_df["distance"].between(low, high)
    ].copy()

    if part.empty:
        continue

    # Выбираем пары ближе к середине диапазона
    midpoint = (low + high) / 2
    part["_priority"] = (
        part["distance"] - midpoint
    ).abs()

    # По возможности из разных видеосерий
    selected = (
        part.sort_values("_priority")
        .drop_duplicates(["class", "shooting"])
        .head(2)
    )

    selected_parts.append(selected)

if selected_parts:

    high_distance_examples = pd.concat(
        selected_parts,
        ignore_index=True
    )

    print(
        "\nВыбрано пар:",
        len(high_distance_examples)
    )

    # 3. Показываем фотографии
    fig, axes = plt.subplots(
        len(high_distance_examples), 2,
        figsize=(10, len(high_distance_examples) * 3.2),
        squeeze=False
    )

    for i, (_, row) in enumerate(
        high_distance_examples.iterrows()
    ):

        for j, filename in enumerate([
            row["prev_filename"],
            row["filename"]
        ]):

            ax = axes[i, j]

            paths = image_index.get(
                (row["class"].lower(), filename), []
            )

            img = (
                cv2.imread(str(paths[0]))
                if paths else None
            )

            if img is not None:
                ax.imshow(
                    cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                )

            ax.set_title(
                f"{row['class']} | {row['shooting']}\n"
                f"{filename}\n"
                f"pHash: {row['distance']}",
                fontsize=9
            )

            ax.axis("off")

    plt.tight_layout()
    plt.show()

else:
    print("Пар с расстоянием выше 16 не найдено.")


In [ ]:

import os
from functools import lru_cache

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim

# =========================================================
# 1. Подготавливаем существующие данные
# =========================================================

needed = {"class", "shooting", "filename", "frame_num", "phash"}

assert needed.issubset(redundancy_df.columns), (
    f"Не хватает столбцов: {needed - set(redundancy_df.columns)}"
)

df = redundancy_df.dropna(subset=list(needed)).copy()
df["frame_num"] = pd.to_numeric(df["frame_num"]).astype(int)

df = df.sort_values(
    ["class", "shooting", "frame_num", "filename"]
).reset_index(drop=True)

assert not df.duplicated(["class", "filename"]).any(), (
    "Обнаружены повторяющиеся идентификаторы изображений"
)

assert not df.duplicated(
    ["class", "shooting", "frame_num"]
).any(), "Повторяется номер кадра внутри видеосерии"

assert df.groupby(["class", "shooting"]).ngroups == 19, (
    "Ожидалось 19 видеосерий. Проверь группировку."
)

R = df.to_dict("records")

for r in R:
    r["hash_int"] = int(str(r["phash"]), 16)

# =========================================================
# 2. Однократно загружаем уменьшенные изображения
# =========================================================

IMG = []

for r in R:

    paths = image_index.get(
        (str(r["class"]).lower(), str(r["filename"])), []
    )

    if not paths:
        raise FileNotFoundError(
            f"Нет пути: {r['class']}/{r['filename']}"
        )

    path = str(paths[0])
    im = cv2.imread(path)

    if im is None:
        raise ValueError(f"Не удалось открыть: {path}")

    small = cv2.resize(
        im, (128, 128), interpolation=cv2.INTER_AREA
    )

    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(small, cv2.COLOR_BGR2HSV)

    hist = cv2.calcHist(
        [hsv], [0, 1], None,
        [12, 8], [0, 180, 0, 256]
    )

    hist = cv2.normalize(
        hist, None, alpha=1.0, norm_type=cv2.NORM_L1
    ).astype("float32")

    IMG.append((gray, hist))

# =========================================================
# 3. Считаем три показателя сходства
# =========================================================

@lru_cache(maxsize=70000)
def _metrics(i, j):

    g1, h1 = IMG[i]
    g2, h2 = IMG[j]

    pdist = (
        R[i]["hash_int"] ^ R[j]["hash_int"]
    ).bit_count()

    full = ssim(g1, g2, data_range=255)

    center = ssim(
        g1[16:112, 16:112],
        g2[16:112, 16:112],
        data_range=255
    )

    color = 1 - cv2.compareHist(
        h1, h2, cv2.HISTCMP_BHATTACHARYYA
    )

    return (
        int(pdist),
        float(full),
        float(center),
        float(color)
    )

def M(i, j):
    return _metrics(min(i, j), max(i, j))

by_shooting = {}

for i, r in enumerate(R):
    key = (r["class"], r["shooting"])
    by_shooting.setdefault(key, []).append(i)

# =========================================================
# 4. SSIM всех действительно соседних кадров
# =========================================================

adj = []

for key, ids in by_shooting.items():

    for a, b in zip(ids[:-1], ids[1:]):

        if R[b]["frame_num"] - R[a]["frame_num"] != 1:
            continue

        pdist, full, center, color = M(a, b)

        adj.append({
            "class_name": key[0],
            "shooting": key[1],
            "prev_filename": R[a]["filename"],
            "filename": R[b]["filename"],
            "pHash_distance": pdist,
            "ssim_full": full,
            "ssim_center": center,
            "color_similarity": color
        })

adj_df = pd.DataFrame(adj)

# =========================================================
# 5. Две предварительные стратегии сходства
# =========================================================

def near_duplicate(values, mode):

    dist, full, center, color = values

    if mode == "strict":
        return (
            (
                dist <= 10
                and full >= 0.95
                and center >= 0.93
                and color >= 0.93
            )
            or (
                full >= 0.985
                and center >= 0.97
                and color >= 0.96
            )
        )

    return (
        (
            dist <= 16
            and full >= 0.90
            and center >= 0.88
            and color >= 0.88
        )
        or (
            full >= 0.975
            and center >= 0.95
            and color >= 0.94
        )
    )

# =========================================================
# 6. Отбор представителей каждой видеосерии
# =========================================================

def select(mode):

    results = []

    for (cls, shoot), ids in by_shooting.items():

        kept = []

        for i in ids:

            # Ищем похожих представителей во всей серии
            near = [
                k for k in kept
                if (
                    R[i]["hash_int"] ^ R[k]["hash_int"]
                ).bit_count() <= 20
            ]

            near = sorted(
                near,
                key=lambda k: (
                    R[i]["hash_int"] ^ R[k]["hash_int"]
                ).bit_count()
            )[:20]

            # Также проверяем последних сохранённых
            # представителей независимо от pHash
            candidates = list(dict.fromkeys(
                list(reversed(kept[-8:])) + near
            ))

            match = None

            for k in candidates:

                vals = M(k, i)

                if near_duplicate(vals, mode):
                    match = (k, vals)
                    break

            if match is not None:

                k, vals = match
                decision = "suggest_skip"
                matched = R[k]["filename"]

            else:

                kept.append(i)
                decision = "keep"
                matched = ""
                vals = (np.nan,) * 4

            results.append({
                "class_name": cls,
                "shooting": shoot,
                "filename": R[i]["filename"],
                "frame_num": R[i]["frame_num"],
                "decision": decision,
                "matched_filename": matched,
                "pHash_distance": vals[0],
                "ssim_full": vals[1],
                "ssim_center": vals[2],
                "color_similarity": vals[3]
            })

    return pd.DataFrame(results)

strict_df = select("strict")
balanced_df = select("balanced")

# =========================================================
# 7. Сводная статистика по видеосериям
# =========================================================

def totals(result, mode):

    z = (
        result.groupby(["class_name", "shooting"])["decision"]
        .agg(
            n_images="size",
            suggest_skip=lambda x: (
                x == "suggest_skip"
            ).sum()
        )
        .reset_index()
    )

    z["kept"] = z["n_images"] - z["suggest_skip"]

    z["suggest_skip_pct"] = (
        100 * z["suggest_skip"] / z["n_images"]
    ).round(1)

    z.insert(0, "policy", mode)

    return z

summary = pd.concat([
    totals(strict_df, "strict"),
    totals(balanced_df, "balanced")
], ignore_index=True)

print("Фотографий в рабочей выборке:", len(R))
print("Видеосерий:", len(by_shooting))
print("Соседних пар:", len(adj_df))

print("\nПроцентили показателей сходства:")

display(
    adj_df[
        [
            "pHash_distance",
            "ssim_full",
            "ssim_center",
            "color_similarity"
        ]
    ].quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.95]).round(3)
)

print("\nРезультаты предварительного сокращения:")

for name, result in [
    ("strict", strict_df),
    ("balanced", balanced_df)
]:

    skip = (result["decision"] == "suggest_skip").sum()

    print(
        f"{name}: оставить {len(result) - skip}, "
        f"кандидатов на исключение {skip} "
        f"({100 * skip / len(result):.1f}%)"
    )

print("\nСтатистика по видеосериям:")

display(
    summary.sort_values(
        ["policy", "class_name", "shooting"]
    )
)

print("\nСтатистика по классам:")

display(
    summary.groupby(
        ["policy", "class_name"]
    )[["n_images", "suggest_skip", "kept"]].sum()
)

# =========================================================
# 8. Графики SSIM и pHash
# =========================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(
    adj_df["ssim_full"],
    bins=30,
    edgecolor="white"
)

axes[0].set(
    xlabel="SSIM",
    ylabel="Количество соседних пар",
    title="Распределение SSIM"
)

axes[1].scatter(
    adj_df["pHash_distance"],
    adj_df["ssim_full"],
    s=9,
    alpha=0.28
)

axes[1].set(
    xlabel="pHash расстояние",
    ylabel="SSIM",
    title="Согласованность pHash и SSIM"
)

plt.tight_layout()
plt.show()

# =========================================================
# 9. Автоматически собираем спорные пары
# =========================================================

ss = strict_df[
    strict_df["decision"] == "suggest_skip"
].sort_values("ssim_full")

ss = (
    ss.groupby("class_name", group_keys=False)
    .head(2)
    .assign(category="strict borderline")
)

bal = balanced_df[
    balanced_df["decision"] == "suggest_skip"
].copy()

strict_keepers = set(zip(
    strict_df.loc[
        strict_df.decision == "keep", "class_name"
    ],
    strict_df.loc[
        strict_df.decision == "keep", "filename"
    ]
))

bal = bal[
    [
        (c, f) in strict_keepers
        for c, f in zip(bal.class_name, bal.filename)
    ]
]

bal = (
    bal.sort_values("ssim_full")
    .groupby("class_name", group_keys=False)
    .head(2)
    .assign(category="balanced only")
)

conf = (
    adj_df[adj_df.pHash_distance >= 20]
    .sort_values("ssim_full", ascending=False)
    .head(4)
)

conf = conf.rename(
    columns={"prev_filename": "matched_filename"}
).assign(category="high pHash")

cols = [
    "class_name", "shooting",
    "matched_filename", "filename",
    "pHash_distance", "ssim_full",
    "ssim_center", "color_similarity",
    "category"
]

audit = (
    pd.concat(
        [ss[cols], bal[cols], conf[cols]],
        ignore_index=True
    )
    .drop_duplicates(
        ["class_name", "filename", "matched_filename"]
    )
)

print("\nПар для итоговой визуальной проверки:", len(audit))

# =========================================================
# 10. Показываем спорные пары
# =========================================================

def show_audit(audit, rows_per_page=8):

    for start in range(0, len(audit), rows_per_page):

        page = audit.iloc[start:start + rows_per_page]

        fig, axes = plt.subplots(
            len(page), 2,
            figsize=(9, 2.6 * len(page)),
            squeeze=False
        )

        for j, (_, a) in enumerate(page.iterrows()):

            for side, fn in enumerate([
                a["matched_filename"],
                a["filename"]
            ]):

                ax = axes[j, side]

                paths = image_index.get(
                    (str(a["class_name"]).lower(), str(fn)),
                    []
                )

                im = (
                    cv2.imread(str(paths[0]))
                    if paths else None
                )

                if im is not None:
                    ax.imshow(
                        cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
                    )

                ax.set_title(
                    f"{a['class_name']}: {fn}\n"
                    f"{a['category']} | "
                    f"pH={a['pHash_distance']:.0f} | "
                    f"SSIM={a['ssim_full']:.3f}",
                    fontsize=8
                )

                ax.axis("off")

        plt.tight_layout()
        plt.show()

show_audit(audit)

# =========================================================
# 11. Сохраняем результаты, но НЕ меняем датасет
# =========================================================

out_dir = (
    "/content/drive/MyDrive/Magnum/redundancy_results"
    if os.path.isdir("/content/drive/MyDrive")
    else "/content/magnum_redundancy_results"
)

os.makedirs(out_dir, exist_ok=True)

outputs = {
    "neighbor_ssim.csv": adj_df,
    "strict_proposal.csv": strict_df,
    "balanced_proposal.csv": balanced_df,
    "series_summary.csv": summary,
    "audit_pairs.csv": audit
}

for filename, table in outputs.items():
    table.to_csv(
        os.path.join(out_dir, filename),
        index=False
    )

print("\nРезультаты сохранены:", out_dir)
print("Кеш сравнений:", _metrics.cache_info())
print("Исходный датасет не изменён.")


In [ ]:

import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image
from tqdm.auto import tqdm
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

# =====================================================
# 1. Проверяем предыдущие результаты
# =====================================================

required = [
    "R", "by_shooting", "M", "image_index",
    "strict_df", "balanced_df"
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        "Не найдены переменные: " + ", ".join(missing)
        + ". Нужны результаты предыдущего анализа."
    )

# Сохраняем всё локально, БЕЗ Google Диска

save_dir = "/content/magnum_redundancy_results"
os.makedirs(save_dir, exist_ok=True)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Устройство:", device)
print("Фотографий:", len(R))
print("Видеосерий:", len(by_shooting))
print("Папка результатов:", save_dir)

# =====================================================
# 2. Загружаем предобученную ResNet18
# =====================================================

model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Убираем классификатор.
# Получаем 512 признаков вместо предсказания класса.

model.fc = nn.Identity()
model = model.to(device).eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# =====================================================
# 3. Извлекаем признаки всех фотографий
# =====================================================

features = []

batch_size = 64 if device.type == "cuda" else 16

for start in tqdm(
    range(0, len(R), batch_size),
    desc="Извлечение CNN-признаков"
):

    batch = []

    for r in R[start:start + batch_size]:

        paths = image_index.get(
            (str(r["class"]).lower(), r["filename"]),
            []
        )

        if not paths:
            raise FileNotFoundError(r["filename"])

        with Image.open(str(paths[0])) as im:
            batch.append(
                transform(im.convert("RGB"))
            )

    x = torch.stack(batch).to(device)

    with torch.inference_mode():
        z = model(x)
        z = F.normalize(z, dim=1)

    features.append(z.cpu().numpy())

E = np.concatenate(features).astype(np.float32)

assert E.shape == (len(R), 512)

np.save(
    os.path.join(save_dir, "resnet18_embeddings.npy"),
    E
)

manifest = pd.DataFrame([
    {
        "row_id": i,
        "class": r["class"],
        "shooting": r["shooting"],
        "filename": r["filename"]
    }
    for i, r in enumerate(R)
])

manifest.to_csv(
    os.path.join(save_dir, "embedding_manifest.csv"),
    index=False
)

print("\nПризнаки всех изображений извлечены.")
print("Размер матрицы:", E.shape)

# =====================================================
# 4. Анализ сходства кадров
# =====================================================

neighbor_sims = []
distant_sims = []

for key, ids in by_shooting.items():

    A = E[ids]
    S = A @ A.T

    nums = np.array([
        R[i]["frame_num"] for i in ids
    ])

    for j in range(1, len(ids)):

        if nums[j] - nums[j - 1] == 1:
            neighbor_sims.append(S[j - 1, j])

    upper = np.triu_indices(len(ids), k=1)

    gaps = np.abs(
        nums[upper[0]] - nums[upper[1]]
    )

    distant_sims.extend(
        S[upper][gaps >= 20].tolist()
    )

print("\nСходство соседних кадров:")

display(
    pd.Series(neighbor_sims)
    .quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99])
    .round(4)
)

print("\nСходство кадров с интервалом >= 20:")

display(
    pd.Series(distant_sims)
    .quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99])
    .round(4)
)

# =====================================================
# 5. Настройки двух стратегий
# =====================================================

policies = {
    "cnn_strict": {
        "cos": 0.995,
        "ssim": 0.85,
        "phash": 8,
        "color": 0.90
    },
    "cnn_balanced": {
        "cos": 0.985,
        "ssim": 0.75,
        "phash": 12,
        "color": 0.88
    }
}

# =====================================================
# 6. Отбор представителей всех видеосерий
# =====================================================

def run_selection(policy):

    records = []

    for (cls, shoot), ids in tqdm(
        by_shooting.items(),
        desc="Поиск похожих кадров",
        leave=False
    ):

        kept = []

        for i in ids:

            best_k = None
            best_cos = np.nan
            match = None

            if kept:

                sims = E[kept] @ E[i]
                order = np.argsort(sims)[::-1]

                best_k = kept[order[0]]
                best_cos = float(sims[order[0]])

                # Проверяем 30 ближайших по CNN
                # представителей во всей видеосерии

                for pos in order[:30]:

                    k = kept[pos]
                    cosine = float(sims[pos])

                    if cosine < policy["cos"]:
                        break

                    ph, sf, sc, color = M(i, k)

                    similar = (
                        (
                            sf >= policy["ssim"]
                            or ph <= policy["phash"]
                        )
                        and color >= policy["color"]
                    )

                    if similar:

                        match = (
                            k, cosine, ph,
                            sf, sc, color
                        )

                        break

            if match is None:

                kept.append(i)
                decision = "keep"

                representative = (
                    R[best_k]["filename"]
                    if best_k is not None else ""
                )

                similarity = best_cos
                ph = sf = sc = color = np.nan

            else:

                k, similarity, ph, sf, sc, color = match

                representative = R[k]["filename"]
                decision = "suggest_skip"

            records.append({
                "class_name": cls,
                "shooting": shoot,
                "filename": R[i]["filename"],
                "frame_num": R[i]["frame_num"],
                "decision": decision,
                "matched_filename": representative,
                "cosine": similarity,
                "pHash_distance": ph,
                "ssim_full": sf,
                "ssim_center": sc,
                "color_similarity": color
            })

    return pd.DataFrame(records)

cnn_results = {}

for name, policy in policies.items():

    print("\nАнализ:", name)

    result = run_selection(policy)

    cnn_results[name] = result

    result.to_csv(
        os.path.join(save_dir, f"{name}.csv"),
        index=False
    )

# =====================================================
# 7. Сравниваем четыре стратегии сокращения
# =====================================================

comparison = []

for name, result in [
    ("old_strict", strict_df),
    ("old_balanced", balanced_df),
    *cnn_results.items()
]:

    for cls, part in result.groupby("class_name"):

        skipped = (
            part["decision"] == "suggest_skip"
        ).sum()

        comparison.append({
            "policy": name,
            "class": cls,
            "total": len(part),
            "suggest_skip": int(skipped),
            "kept": len(part) - skipped
        })

comparison_df = pd.DataFrame(comparison)

print("\nСравнение по классам:")

display(comparison_df)

print("\nОБЩИЕ ИТОГИ:")

display(
    comparison_df.groupby("policy")[
        ["total", "suggest_skip", "kept"]
    ].sum()
)

comparison_df.to_csv(
    os.path.join(save_dir, "cnn_comparison.csv"),
    index=False
)

# =====================================================
# 8. Подбираем фотографии для визуальной проверки
# =====================================================

audit_parts = []

frame_lookup = {
    (r["class"], r["filename"]): r["frame_num"]
    for r in R
}

for policy_name, result in cnn_results.items():

    skipped = result[
        result.decision == "suggest_skip"
    ].copy()

    if skipped.empty:
        continue

    skipped["frame_gap"] = [
        abs(
            int(row.frame_num)
            - frame_lookup[
                (row.class_name, row.matched_filename)
            ]
        )
        for row in skipped.itertuples()
    ]

    # Наиболее спорные кандидаты
    borderline = (
        skipped.sort_values("cosine")
        .groupby("class_name", group_keys=False)
        .head(1)
    )

    # Повторы, находящиеся далеко друг от друга
    distant = (
        skipped.sort_values(
            "frame_gap", ascending=False
        )
        .groupby("class_name", group_keys=False)
        .head(1)
    )

    selected = pd.concat([
        borderline, distant
    ]).drop_duplicates(
        ["class_name", "filename", "matched_filename"]
    )

    selected["audit_type"] = policy_name

    audit_parts.append(selected)

# Ищем очень похожие кадры,
# которые CNN всё-таки решил сохранить

balanced = cnn_results["cnn_balanced"]

potential_misses = balanced[
    (balanced.decision == "keep")
    & (balanced.cosine >= 0.985)
    & (balanced.matched_filename != "")
].copy()

potential_misses = (
    potential_misses.sort_values(
        "cosine", ascending=False
    )
    .groupby("class_name", group_keys=False)
    .head(2)
)

potential_misses["audit_type"] = "potential_miss"

audit_parts.append(potential_misses)

audit_cnn_df = (
    pd.concat(audit_parts, ignore_index=True)
    .drop_duplicates(
        ["class_name", "filename", "matched_filename"]
    )
)

audit_cnn_df.to_csv(
    os.path.join(save_dir, "cnn_audit.csv"),
    index=False
)

print("\nПар для визуальной проверки:", len(audit_cnn_df))

# =====================================================
# 9. Показываем спорные фотографии
# =====================================================

def show_cnn_audit(table, page_size=6):

    for start in range(0, len(table), page_size):

        page = table.iloc[start:start + page_size]

        fig, axes = plt.subplots(
            len(page), 2,
            figsize=(9, 2.8 * len(page)),
            squeeze=False
        )

        for j, row in enumerate(page.itertuples()):

            for side, fn in enumerate([
                row.matched_filename,
                row.filename
            ]):

                ax = axes[j, side]

                paths = image_index.get(
                    (row.class_name.lower(), fn), []
                )

                im = (
                    cv2.imread(str(paths[0]))
                    if paths else None
                )

                if im is not None:

                    ax.imshow(
                        cv2.cvtColor(
                            im, cv2.COLOR_BGR2RGB
                        )
                    )

                ax.set_title(
                    f"{row.class_name} | {fn}\n"
                    f"{row.audit_type} | "
                    f"cos={row.cosine:.4f}",
                    fontsize=8
                )

                ax.axis("off")

        plt.tight_layout()
        plt.show()

show_cnn_audit(audit_cnn_df)

# =====================================================
# 10. Создаём архив для будущего скачивания
# =====================================================

archive = shutil.make_archive(
    "/content/Magnum_results",
    "zip",
    save_dir
)

print("\n===================================")
print("АНАЛИЗ ЗАВЕРШЁН")
print("===================================")
print("Исходный датасет НЕ изменён.")
print("Результаты:", save_dir)
print("Архив:", archive)
print("Архив можно скачать позже.")


In [ ]:

from google.colab import files

files.download("/content/Magnum_results.zip")


In [ ]:

import os
import shutil
import pandas as pd
import numpy as np

# =====================================================
# 1. Проверяем наличие данных
# =====================================================

if "phash_df" not in globals():
    raise RuntimeError("Не найдена переменная phash_df")

if "quality_flags_df" not in globals():
    raise RuntimeError("Не найдена quality_flags_df")

save_dir = "/content/magnum_redundancy_results"
os.makedirs(save_dir, exist_ok=True)

cnn_path = os.path.join(save_dir, "cnn_balanced.csv")

if os.path.exists(cnn_path):
    cnn = pd.read_csv(cnn_path)
elif "cnn_results" in globals():
    cnn = cnn_results["cnn_balanced"].copy()
else:
    raise RuntimeError(
        "Не найден результат CNN Balanced"
    )

# =====================================================
# 2. Строим единый список уникальных фотографий
# =====================================================

required = {"class", "filename", "shooting"}

assert required.issubset(phash_df.columns), (
    "В phash_df отсутствуют необходимые столбцы"
)

manifest_df = phash_df.copy()

def make_keys(table, class_col="class"):
    classes = (
        table[class_col].astype(str)
        .str.strip().str.casefold()
    )
    filenames = (
        table["filename"].astype(str).str.strip()
    )
    return list(zip(classes, filenames))

manifest_keys = make_keys(manifest_df)

if len(set(manifest_keys)) != len(manifest_keys):
    raise ValueError(
        "В phash_df обнаружены повторные ключи "
        "(class, filename). Нужна сверка."
    )

# Определяем кадры видеосерий по имени файла

manifest_df["is_video_frame"] = (
    manifest_df["filename"].astype(str)
    .str.contains(
        r"_frame_\d+\.[^.]+$",
        case=False,
        regex=True,
        na=False
    )
)

manifest_df["photo_type"] = np.where(
    manifest_df["is_video_frame"],
    "video",
    "single"
)

# =====================================================
# 3. Сопоставляем кандидатов по качеству
# =====================================================

quality_keys = set(make_keys(quality_flags_df))
all_keys = set(manifest_keys)

unknown_quality = quality_keys - all_keys

if unknown_quality:
    raise ValueError(
        "Есть отметки качества без соответствующих "
        f"фотографий: {list(unknown_quality)[:5]}"
    )

manifest_df["quality_candidate"] = [
    key in quality_keys for key in manifest_keys
]

# =====================================================
# 4. Сопоставляем результаты CNN Balanced
# =====================================================

required_cnn = {
    "class_name", "filename",
    "decision", "matched_filename"
}

assert required_cnn.issubset(cnn.columns), (
    "В cnn_balanced.csv отсутствуют столбцы"
)

cnn_keys_list = make_keys(cnn, "class_name")
cnn_keys = set(cnn_keys_list)

if len(cnn_keys_list) != len(cnn_keys):
    raise ValueError("Повторяющиеся ключи в CNN-таблице")

if not set(cnn["decision"]).issubset(
    {"keep", "suggest_skip"}
):
    raise ValueError("Неизвестные решения CNN")

redundancy_keys = set(
    make_keys(
        cnn[cnn["decision"] == "suggest_skip"],
        "class_name"
    )
)

# Проверяем, что каждый исключаемый кадр
# ссылается на сохранённого представителя

cnn_kept_keys = set(
    make_keys(
        cnn[cnn["decision"] == "keep"],
        "class_name"
    )
)

skipped = cnn[cnn["decision"] == "suggest_skip"]

representative_keys = set(zip(
    skipped["class_name"].astype(str)
    .str.strip().str.casefold(),
    skipped["matched_filename"].astype(str).str.strip()
))

if not representative_keys.issubset(cnn_kept_keys):
    raise ValueError(
        "Некоторые кандидаты ссылаются "
        "на отсутствующего представителя"
    )

manifest_df["redundancy_checked"] = [
    key in cnn_keys for key in manifest_keys
]

manifest_df["redundancy_candidate"] = [
    key in redundancy_keys for key in manifest_keys
]

# =====================================================
# 5. Проверяем полноту и согласованность
# =====================================================

clean_video_keys = {
    key for key, video, bad in zip(
        manifest_keys,
        manifest_df["is_video_frame"],
        manifest_df["quality_candidate"]
    )
    if video and not bad
}

video_groups = (
    manifest_df.loc[
        manifest_df["is_video_frame"],
        ["class", "shooting"]
    ]
    .drop_duplicates()
)

checks = {
    "Всего уникальных фотографий":
        (len(manifest_df), 3879),

    "Одиночных фотографий":
        (int((~manifest_df["is_video_frame"]).sum()), 185),

    "Видеосерий":
        (len(video_groups), 19),

    "Кандидатов по качеству":
        (len(quality_keys), 156),

    "Кадров, рассмотренных CNN":
        (len(cnn_keys), 3540),

    "Кандидатов CNN Balanced":
        (len(redundancy_keys), 1033)
}

check_df = pd.DataFrame([
    {
        "check": name,
        "actual": actual,
        "expected": expected,
        "ok": actual == expected
    }
    for name, (actual, expected) in checks.items()
])

print("СВЕРКА ДАТАСЕТА:")
display(check_df)

if not check_df["ok"].all():
    raise ValueError(
        "Некоторые количества не совпали. "
        "Пока не используем результаты для разбиения."
    )

if clean_video_keys != cnn_keys:
    raise ValueError(
        "CNN-таблица не полностью соответствует "
        "кадрам видеосерий после фильтра качества."
    )

if (
    manifest_df["quality_candidate"]
    & manifest_df["redundancy_candidate"]
).any():
    raise ValueError(
        "Обнаружено пересечение между исключениями "
        "по качеству и CNN. Нужна проверка."
    )

# =====================================================
# 6. Создаём две версии будущего набора
# =====================================================

# Сценарий A: только фильтрация качества
manifest_df["use_quality_only"] = (
    ~manifest_df["quality_candidate"]
)

# Сценарий B: качество + CNN Balanced
manifest_df["use_quality_and_cnn"] = (
    ~manifest_df["quality_candidate"]
    & ~manifest_df["redundancy_candidate"]
)

# Это лишь предварительные флаги.
# Никакие файлы не удаляем.

# =====================================================
# 7. Отмечаем принадлежность к видеосерии
# =====================================================

manifest_df["video_series_id"] = pd.NA

v = manifest_df["is_video_frame"]

manifest_df.loc[v, "video_series_id"] = (
    manifest_df.loc[v, "class"].astype(str)
    + "__"
    + manifest_df.loc[v, "shooting"].astype(str)
)

# Для одиночных фото группы пока не назначаем.
# Их происхождение необходимо учитывать отдельно
# при последующем Train/Val/Test.

# =====================================================
# 8. Итоговая статистика по классам
# =====================================================

class_summary_df = (
    manifest_df.groupby("class")
    .agg(
        total=("filename", "size"),
        video=("is_video_frame", "sum"),
        single=("is_video_frame", lambda x: (~x).sum()),
        quality_candidates=("quality_candidate", "sum"),
        cnn_candidates=("redundancy_candidate", "sum"),
        after_quality=("use_quality_only", "sum"),
        after_both=("use_quality_and_cnn", "sum")
    )
    .reset_index()
)

print("\nИТОГОВАЯ ТАБЛИЦА ПО КЛАССАМ:")
display(class_summary_df)

print("\nОБЩИЕ ИТОГИ:")

print("Всего фотографий:", len(manifest_df))
print(
    "После фильтра качества:",
    int(manifest_df["use_quality_only"].sum())
)
print(
    "После качества + CNN:",
    int(manifest_df["use_quality_and_cnn"].sum())
)

print(
    "Одиночных фотографий с отметкой качества:",
    int((
        ~manifest_df["is_video_frame"]
        & manifest_df["quality_candidate"]
    ).sum())
)

# =====================================================
# 9. Сохраняем результаты и обновляем архив
# =====================================================

manifest_df.to_csv(
    os.path.join(save_dir, "master_manifest.csv"),
    index=False
)

class_summary_df.to_csv(
    os.path.join(save_dir, "final_class_summary.csv"),
    index=False
)

check_df.to_csv(
    os.path.join(save_dir, "data_integrity_checks.csv"),
    index=False
)

archive_path = shutil.make_archive(
    "/content/Magnum_results",
    "zip",
    save_dir
)

print("\n====================================")
print("СВЕРКА ЗАВЕРШЕНА")
print("====================================")
print("Единый реестр:", len(manifest_df), "фотографий")
print("Папка результатов:", save_dir)
print("Обновлённый архив:", archive_path)
print("Исходные фотографии не изменены.")


In [ ]:

import pandas as pd

# Используем готовый единый реестр

df = manifest_df.copy()

video = df[df["is_video_frame"]].copy()
single = df[~df["is_video_frame"]].copy()

# 1. Определяем дату видеосъёмки
video["shoot_date"] = (
    video["shooting"].astype(str)
    .str.extract(r"(\d{8})")[0]
)

if video["shoot_date"].isna().any():
    raise ValueError("Не удалось определить дату некоторых съёмок")

# 2. Сводка по независимым датам и сериям

date_groups = (
    video.groupby(["class", "shoot_date"])
    .agg(
        video_series=("shooting", "nunique"),
        frames=("filename", "size"),
        shooting_ids=(
            "shooting",
            lambda x: ", ".join(
                sorted(x.astype(str).unique())
            )
        )
    )
    .reset_index()
)

print("ГРУППЫ ВИДЕОСЪЁМОК ПО ДАТАМ:")
display(date_groups)

# 3. Сколько дат съёмки у каждого класса

print("\nКОЛИЧЕСТВО ДАТ СЪЁМКИ ПО КЛАССАМ:")

display(
    date_groups.groupby("class")
    .agg(
        shooting_dates=("shoot_date", "nunique"),
        video_series=("video_series", "sum"),
        video_frames=("frames", "sum")
    )
    .reset_index()
)

# 4. Проверяем происхождение одиночных фотографий

print("\nОДИНОЧНЫЕ ФОТОГРАФИИ:")
display(
    single.groupby("class")
    .agg(
        photos=("filename", "size"),
        shooting_values=("shooting", "nunique")
    )
    .reset_index()
)

print("\nПРИМЕРЫ ИМЁН ОДИНОЧНЫХ ФОТО:")

display(
    single.groupby("class", group_keys=False)
    .head(5)[
        ["class", "filename", "shooting"]
    ]
)

print("\nПроверка завершена. Фотографии не изменены.")


In [ ]:

import os
import shutil
import pandas as pd

# Используем существующий реестр
split_base_df = manifest_df.copy()

# Извлекаем дату из имени каждой фотографии
split_base_df["capture_date"] = (
    split_base_df["filename"]
    .astype(str)
    .str.extract(r"^(\d{8})_")[0]
)

assert split_base_df["capture_date"].notna().all(), (
    "Есть фотографии без даты в имени"
)

# Консервативная группа: класс + дата
split_base_df["conservative_group"] = (
    split_base_df["class"].astype(str)
    + "__"
    + split_base_df["capture_date"]
)

# Даты, на которые есть видеосерии
video_dates = set(zip(
    split_base_df.loc[
        split_base_df.is_video_frame, "class"
    ],
    split_base_df.loc[
        split_base_df.is_video_frame, "capture_date"
    ]
))

# Проверяем одиночные снимки на совпадение
# класса и даты с видеосериями
split_base_df["single_on_video_date"] = [
    (not is_video) and ((cls, date) in video_dates)
    for cls, date, is_video in zip(
        split_base_df["class"],
        split_base_df["capture_date"],
        split_base_df["is_video_frame"]
    )
]

# Сводка всех фотографий по группам
group_report = (
    split_base_df
    .groupby(["class", "capture_date", "conservative_group"])
    .agg(
        total=("filename", "size"),
        video=("is_video_frame", "sum"),
        quality_candidates=("quality_candidate", "sum"),
        cnn_candidates=("redundancy_candidate", "sum")
    )
    .reset_index()
)

group_report["single"] = (
    group_report["total"] - group_report["video"]
)

print("ВСЕ ГРУППЫ ПО КЛАССАМ И ДАТАМ:")
display(group_report)

# Проверяем возможность трёх независимых частей
feasibility = (
    group_report.groupby("class")
    .agg(
        groups=("conservative_group", "nunique"),
        total=("total", "sum")
    )
    .reset_index()
)

feasibility["three_way_possible"] = (
    feasibility["groups"] >= 3
)

print("\nВОЗМОЖНОСТЬ TRAIN / VAL / TEST:")
display(feasibility)

single = split_base_df[
    ~split_base_df["is_video_frame"]
]

print("\nОДИНОЧНЫЕ ФОТОГРАФИИ:")
print("Всего:", len(single))
print(
    "Сняты в даты имеющихся видеосерий:",
    int(single["single_on_video_date"].sum())
)
print(
    "Сняты в другие даты:",
    int((~single["single_on_video_date"]).sum())
)

# Сохраняем результаты
save_dir = "/content/magnum_redundancy_results"
os.makedirs(save_dir, exist_ok=True)

split_base_df.to_csv(
    os.path.join(save_dir, "manifest_with_groups.csv"),
    index=False
)

group_report.to_csv(
    os.path.join(save_dir, "date_group_report.csv"),
    index=False
)

feasibility.to_csv(
    os.path.join(save_dir, "split_feasibility.csv"),
    index=False
)

shutil.make_archive(
    "/content/Magnum_results",
    "zip",
    save_dir
)

print("\nГруппы сохранены.")
print("Исходный датасет не изменён.")


In [ ]:

import os
import shutil
import pandas as pd
from google.colab import files

save_dir = "/content/magnum_redundancy_results"
os.makedirs(save_dir, exist_ok=True)

# Сохраняем важные таблицы,
# которые раньше существовали только в памяти

tables = [
    "phash_df",
    "quality_flags_df",
    "video_quality_df",
    "redundancy_df",
    "manifest_df",
    "split_base_df",
    "group_report",
    "feasibility"
]

for name in tables:
    obj = globals().get(name)

    if isinstance(obj, pd.DataFrame):
        path = os.path.join(save_dir, f"{name}.csv")
        obj.to_csv(path, index=False)
        print("Сохранено:", name)

# Собираем все результаты в один архив

archive = shutil.make_archive(
    "/content/Magnum_FULL_BACKUP",
    "zip",
    save_dir
)

print("\nРазмер архива:",
      round(os.path.getsize(archive) / 1024**2, 2),
      "МБ")

print("Архив готов:", archive)

# Скачиваем на Mac

files.download(archive)


In [ ]:

import os
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from google.colab import files

# 1. Загружаем резервную копию с компьютера
print("Выбери Magnum_FULL_BACKUP.zip")

uploaded = files.upload()

zip_names = [
    name for name in uploaded
    if name.endswith(".zip")
]

if len(zip_names) != 1:
    raise ValueError("Необходимо загрузить один ZIP-архив")

# 2. Распаковываем результаты
save_dir = Path("/content/magnum_redundancy_results")
save_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_names[0], "r") as z:
    z.extractall(save_dir)

print("\nАрхив распакован.")

# 3. Восстанавливаем таблицы
phash_df = pd.read_csv(save_dir / "phash_df.csv")
quality_flags_df = pd.read_csv(
    save_dir / "quality_flags_df.csv"
)
video_quality_df = pd.read_csv(
    save_dir / "video_quality_df.csv"
)
redundancy_df = pd.read_csv(
    save_dir / "redundancy_df.csv"
)
manifest_df = pd.read_csv(
    save_dir / "manifest_df.csv"
)
split_base_df = pd.read_csv(
    save_dir / "split_base_df.csv",
    dtype={"capture_date": "string"}
)
group_report = pd.read_csv(
    save_dir / "group_report.csv"
)
feasibility = pd.read_csv(
    save_dir / "feasibility.csv"
)

# 4. Восстанавливаем CNN-результаты
cnn_results = {
    "cnn_strict": pd.read_csv(
        save_dir / "cnn_strict.csv"
    ),
    "cnn_balanced": pd.read_csv(
        save_dir / "cnn_balanced.csv"
    )
}

cnn = cnn_results["cnn_balanced"].copy()

# 5. Загружаем сохранённые эмбеддинги
E = np.load(
    save_dir / "resnet18_embeddings.npy"
)

embedding_manifest = pd.read_csv(
    save_dir / "embedding_manifest.csv"
)

# 6. Восстанавливаем порядок фотографий для CNN
sorted_redundancy = (
    redundancy_df.sort_values(
        ["class", "shooting", "frame_num", "filename"]
    )
    .reset_index(drop=True)
)

assert sorted_redundancy[
    ["class", "shooting", "filename"]
].equals(
    embedding_manifest[
        ["class", "shooting", "filename"]
    ]
), "Порядок CNN-признаков не совпадает!"

R = sorted_redundancy.to_dict("records")

for r in R:
    r["hash_int"] = int(str(r["phash"]), 16)

by_shooting = {}

for i, r in enumerate(R):
    key = (r["class"], r["shooting"])
    by_shooting.setdefault(key, []).append(i)

# 7. Проверяем целостность восстановления
assert len(phash_df) == 3879
assert len(manifest_df) == 3879
assert len(quality_flags_df) == 156
assert len(by_shooting) == 19
assert E.shape == (3540, 512)
assert (
    manifest_df["redundancy_candidate"].sum()
    == 1033
)

print("\n" + "=" * 45)
print("ВОССТАНОВЛЕНИЕ УСПЕШНО")
print("=" * 45)
print("Уникальных фото:", len(manifest_df))
print("Кандидатов по качеству:", len(quality_flags_df))
print("Видеосерий:", len(by_shooting))
print("CNN-признаки:", E.shape)
print(
    "Кандидатов по избыточности:",
    int(manifest_df["redundancy_candidate"].sum())
)
print("Группы съёмок восстановлены.")
print("Можно продолжать работу над Train/Val/Test.")


In [ ]:

import pandas as pd
import numpy as np

print("=" * 65)
print("MAGNUM — ПРОВЕРКА СТРУКТУРЫ ДАННЫХ ПЕРЕД SPLIT")
print("=" * 65)

names = [
    "manifest_df",
    "split_base_df",
    "phash_df",
    "quality_flags_df",
    "cnn_results",
    "E"
]

for name in names:
    print(f"\n{'=' * 20} {name} {'=' * 20}")

    if name not in globals():
        print("ПЕРЕМЕННАЯ НЕ НАЙДЕНА")
        continue

    obj = globals()[name]
    print("Тип:", type(obj).__name__)

    if isinstance(obj, pd.DataFrame):
        print("Размер:", obj.shape)
        print("Столбцы:")
        print(list(obj.columns))

        print("\nПервые 2 строки:")
        print(obj.head(2).to_string(
            index=False,
            max_colwidth=35
        ))

    elif isinstance(obj, dict):
        print("Ключи:", list(obj.keys())[:30])

        for k, v in list(obj.items())[:8]:
            print(
                f"  {k}:",
                type(v).__name__,
                "shape =", getattr(v, "shape", None),
                "len =", len(v) if hasattr(v, "__len__") else None
            )

    else:
        print("Shape:", getattr(obj, "shape", None))
        if hasattr(obj, "__len__"):
            print("Length:", len(obj))


print("\n" + "=" * 65)
print("ДОПОЛНИТЕЛЬНЫЕ ТАБЛИЦЫ И НАБОРЫ")
print("=" * 65)

keywords = (
    "split", "group", "quality", "filter",
    "balanced", "selected", "keep",
    "clean", "subset", "cnn"
)

for name, obj in list(globals().items()):
    if name.startswith("_") or name in names:
        continue

    if not any(k in name.lower() for k in keywords):
        continue

    if isinstance(obj, (pd.DataFrame, pd.Series, np.ndarray, list, tuple, set)):
        try:
            print(
                name,
                "| type:", type(obj).__name__,
                "| len:", len(obj),
                "| shape:", getattr(obj, "shape", None)
            )
        except Exception:
            pass


print("\n" + "=" * 65)
print("КЛАССЫ И СЪЁМОЧНЫЕ ГРУППЫ")
print("=" * 65)

for source_name in ["split_base_df", "manifest_df"]:
    if source_name not in globals():
        continue

    df = globals()[source_name]
    if not isinstance(df, pd.DataFrame):
        continue

    label_cols = [
        c for c in df.columns
        if str(c).lower() in (
            "label", "class", "class_name",
            "category", "vegetable"
        )
    ]

    group_cols = [
        c for c in df.columns
        if any(k in str(c).lower()
               for k in ["group", "session", "shoot"])
    ]

    print(f"\nИсточник: {source_name}")
    print("Колонки классов:", label_cols)
    print("Колонки групп:", group_cols)

    if label_cols:
        label_col = label_cols[0]
        print("\nИзображений по классам:")
        print(df[label_col].value_counts().to_string())

        if group_cols:
            group_col = group_cols[0]
            print("\nУникальных групп по классам:")
            print(
                df.groupby(label_col)[group_col]
                  .nunique()
                  .to_string()
            )

print("\nПРОВЕРКА ЗАВЕРШЕНА")


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42
TARGET = np.array([0.70, 0.15, 0.15])
SPLITS = ["train", "val", "test"]

# 1. Берём уже подготовленные данные

base = split_base_df.copy()

required = [
    "class", "filename", "shooting_group",
    "conservative_group", "use_quality_only",
    "use_quality_and_cnn"
]

missing = [c for c in required if c not in base.columns]
assert not missing, f"Отсутствуют столбцы: {missing}"

def boolean_flag(series):
    return series.astype(str).str.lower().str.strip().isin(
        ["true", "1", "yes"]
    )

base["use_quality_only"] = boolean_flag(
    base["use_quality_only"]
)
base["use_quality_and_cnn"] = boolean_flag(
    base["use_quality_and_cnn"]
)

assert not base.duplicated(["class", "filename"]).any()

quality = base.loc[base["use_quality_only"]].copy()

assert len(quality) == 3723, (
    f"Ожидалось 3723 фото, найдено {len(quality)}"
)

assert not (
    base["use_quality_and_cnn"] &
    ~base["use_quality_only"]
).any(), "CNN Balanced содержит фото, исключённые фильтром качества"

print("После фильтра качества:", len(quality))
print("После двух фильтров:",
      int(base["use_quality_and_cnn"].sum()))



# 2. Выбираем группы для каждого класса

quality["split_group"] = ""
quality["group_strategy"] = ""

strategy_report = []

for cls, part in quality.groupby("class"):

    candidates = [
        "conservative_group",
        "shooting_group",
        "shooting"
    ]

    group_counts = {
        col: part[col].nunique(dropna=True)
        for col in candidates
    }

    column = None

    for candidate in candidates:

        values = part[candidate]

        complete = (
            values.notna().all()
            and values.astype(str).str.strip().ne("").all()
        )

        if complete and group_counts[candidate] >= 3:
            column = candidate
            break

    if column is None:
        raise ValueError(
            f"{cls}: даже отдельных съёмок недостаточно "
            "для группового Train/Val/Test."
        )

    quality.loc[part.index, "split_group"] = (
        cls + "||" + part[column].astype(str)
    )

    quality.loc[part.index, "group_strategy"] = column

    strategy_report.append({
        "class": cls,
        "conservative_groups": group_counts["conservative_group"],
        "shooting_groups": group_counts["shooting_group"],
        "individual_shootings": group_counts["shooting"],
        "selected_strategy": column
    })

    print(
        f"{cls}: используем {column}, "
        f"групп = {group_counts[column]}"
    )

    if column == "shooting":
        print(
            "  ВНИМАНИЕ: независимость оценки ограничена; "
            "разные съёмки могут относиться к одной сессии."
        )


# 3. Подбираем разбиение по целым группам
# Нельзя разрывать одну выбранную группу между частями.

def assign_class_groups(part, seed):

    sizes = (
        part.groupby("split_group")
            .size()
            .sort_index()
    )

    keys = sizes.index.to_numpy()
    counts = sizes.to_numpy()

    k = len(keys)
    assert k >= 3

    rng = np.random.default_rng(seed)

    best_loss = np.inf
    best_assignment = None

    for _ in range(350):

        order = rng.permutation(k)
        cumulative = np.cumsum(counts[order])
        total = cumulative[-1]

        for i in range(1, k - 1):

            js = np.arange(i + 1, k)

            train_count = cumulative[i - 1]
            val_counts = cumulative[js - 1] - train_count
            test_counts = total - cumulative[js - 1]

            fractions = np.column_stack([
                np.full(len(js), train_count),
                val_counts,
                test_counts
            ]) / total

            losses = np.sum(
                (fractions - TARGET) ** 2,
                axis=1
            )

            best_j_pos = np.argmin(losses)
            loss = losses[best_j_pos]

            if loss < best_loss:
                best_loss = loss

                j = int(js[best_j_pos])

                best_assignment = {
                    "train": keys[order[:i]].tolist(),
                    "val": keys[order[i:j]].tolist(),
                    "test": keys[order[j:]].tolist()
                }

    return best_assignment


quality["split"] = ""

for class_index, (cls, part) in enumerate(
    quality.groupby("class")
):

    assignment = assign_class_groups(
        part, SEED + class_index
    )

    for split_name, group_keys in assignment.items():

        mask = (
            quality.index.isin(part.index)
            & quality["split_group"].isin(group_keys)
        )

        quality.loc[mask, "split"] = split_name


# 4. Проверки корректности

assert quality["split"].isin(SPLITS).all()
assert len(quality) == 3723

assert (
    quality.groupby("split_group")["split"].nunique().max()
    == 1
), "Обнаружено пересечение выбранных групп!"

assert (
    quality.groupby(["class", "split"])
           .size()
           .unstack(fill_value=0)
           .reindex(columns=SPLITS, fill_value=0)
           .gt(0)
           .all()
           .all()
), "Не все 6 классов представлены во всех частях"

print("\n" + "=" * 65)
print("СТРАТЕГИЯ ГРУППИРОВКИ")
print("=" * 65)

strategy_df = pd.DataFrame(strategy_report)
print(strategy_df.to_string(index=False))

print("\n" + "=" * 65)
print("ИТОГОВОЕ РАЗБИЕНИЕ ПО КЛАССАМ")
print("=" * 65)

distribution = pd.crosstab(
    quality["class"], quality["split"]
).reindex(columns=SPLITS, fill_value=0)

distribution["total"] = distribution.sum(axis=1)

print(distribution.to_string())

print("\nРазмеры выборок:")
print(quality["split"].value_counts().reindex(SPLITS))

print("\nДоли выборок:")
print(
    (quality["split"].value_counts(normalize=True)
     .reindex(SPLITS) * 100).round(2)
)

print("\n" + "=" * 65)
print("ПРОВЕРКА СТРОГИХ КОНСЕРВАТИВНЫХ ГРУПП")
print("=" * 65)

conservative_overlap = (
    quality.groupby(["class", "conservative_group"])["split"]
           .nunique()
)

crossed = conservative_overlap[conservative_overlap > 1]

if crossed.empty:
    print("OK: conservative_group не пересекаются.")
else:
    print("ВНИМАНИЕ: некоторые conservative_group")
    print("попали в разные части:")
    print(crossed.to_string())
    print("\nДля этих классов независимость оценки")
    print("ограничена. Это необходимо указать в отчёте.")

print("\n" + "=" * 65)
print("СОХРАНЕНИЕ")
print("=" * 65)

stage_a_split_df = quality.copy()

stage_a_train_quality = quality[
    quality["split"] == "train"
].copy()

stage_a_val = quality[
    quality["split"] == "val"
].copy()

stage_a_test = quality[
    quality["split"] == "test"
].copy()

drive_root = Path("/content/drive/MyDrive")

if drive_root.exists():
    output_dir = drive_root / "Magnum_StageA"
else:
    output_dir = Path("/content")
    print("Drive не подключён — локальное сохранение.")

output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "stage_a_split.csv"

stage_a_split_df.to_csv(
    output_path, index=False, encoding="utf-8"
)

strategy_df.to_csv(
    output_dir / "stage_a_group_strategy.csv",
    index=False
)

print("Файл сохранён:", output_path)
print("Train:", len(stage_a_train_quality))
print("Validation:", len(stage_a_val))
print("Test:", len(stage_a_test))
print("\nРАЗБИЕНИЕ ЗАВЕРШЕНО")


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42
SPLITS = ["train", "val", "test"]
TARGET = np.array([0.70, 0.15, 0.15])

# 1. Берём исходную подготовленную таблицу
df = split_base_df.copy()

for col in ["use_quality_only", "use_quality_and_cnn"]:
    df[col] = (
        df[col].astype(str).str.strip().str.lower()
        .isin(["true", "1", "yes"])
    )

quality = df[df["use_quality_only"]].copy()

assert len(quality) == 3723
assert df["use_quality_and_cnn"].sum() == 2690
assert quality["shooting"].notna().all()
assert not quality.duplicated(["class", "filename"]).any()

# Все классы разбиваем по конкретным съёмкам
quality["split_group"] = (
    quality["class"].astype(str)
    + "||"
    + quality["shooting"].astype(str)
)

quality["split"] = ""

# 2. Поиск распределения с учётом размеров групп
def find_group_split(part, seed, trials=1200):

    sizes = (
        part.groupby("split_group")
        .size()
        .sort_index()
    )

    groups = sizes.index.to_numpy()
    counts = sizes.to_numpy()
    n_groups = len(groups)
    total = int(counts.sum())

    if n_groups < 3:
        raise ValueError(
            f"Для класса недостаточно съёмок: {n_groups}"
        )

    rng = np.random.default_rng(seed)
    best_score = float("inf")
    best_result = None

    # Мягкое требование к числу проверочных фото
    min_support = max(10, int(np.ceil(total * 0.05)))

    for _ in range(trials):

        order = rng.permutation(n_groups)
        cumulative = np.cumsum(counts[order])

        for i in range(1, n_groups - 1):

            js = np.arange(i + 1, n_groups)

            train_n = cumulative[i - 1]
            val_n = cumulative[js - 1] - train_n
            test_n = total - cumulative[js - 1]

            candidate_counts = np.column_stack([
                np.full(len(js), train_n),
                val_n,
                test_n
            ])

            fractions = candidate_counts / total

            # Близость к целевым пропорциям
            scores = np.sum(
                (fractions - TARGET) ** 2,
                axis=1
            )

            # Штрафуем слишком маленькие выборки
            val_short = np.maximum(
                0, min_support - val_n
            ) / min_support

            test_short = np.maximum(
                0, min_support - test_n
            ) / min_support

            scores += 0.1 * (
                val_short ** 2 + test_short ** 2
            )

            pos = int(np.argmin(scores))

            if scores[pos] < best_score:
                best_score = float(scores[pos])
                j = int(js[pos])

                best_result = {
                    "train": groups[order[:i]].tolist(),
                    "val": groups[order[i:j]].tolist(),
                    "test": groups[order[j:]].tolist()
                }

    return best_result


# 3. Разбиваем каждый класс
for idx, (cls, part) in enumerate(
    quality.groupby("class", sort=True)
):

    assignment = find_group_split(
        part, SEED + idx
    )

    for split_name, groups in assignment.items():
        mask = (
            (quality["class"] == cls)
            & quality["split_group"].isin(groups)
        )
        quality.loc[mask, "split"] = split_name

# 4. Строгие технические проверки
assert quality["split"].isin(SPLITS).all()

assert (
    quality.groupby("split_group")["split"]
    .nunique()
    .max() == 1
), "Одна съёмка попала в разные выборки!"

distribution = pd.crosstab(
    quality["class"],
    quality["split"]
).reindex(columns=SPLITS, fill_value=0)

assert (distribution > 0).all().all(), (
    "Не каждый класс присутствует в Train/Val/Test"
)

print("=" * 65)
print("НОВОЕ РАЗБИЕНИЕ ПО КЛАССАМ")
print("=" * 65)
print(distribution.to_string())

print("\nОбщие размеры:")
print(quality["split"].value_counts().reindex(SPLITS))

print("\nДоли (%):")
print(
    (quality["split"].value_counts(normalize=True)
     .reindex(SPLITS) * 100).round(2)
)

print("\nКоличество съёмок по частям:")
print(
    quality.groupby(["class", "split"])["split_group"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=SPLITS, fill_value=0)
    .to_string()
)

# 5. Проверяем слабые места
print("\n" + "=" * 65)
print("ДИАГНОСТИКА")
print("=" * 65)

small = distribution[
    (distribution["val"] < 20) |
    (distribution["test"] < 20)
]

if small.empty:
    print("OK: в каждом классе >=20 фото в Val и Test.")
else:
    print("ВНИМАНИЕ: малые проверочные выборки:")
    print(small.to_string())

# Считаем пересечения консервативных групп
overlap = (
    quality.groupby(["class", "conservative_group"])["split"]
    .nunique()
)

overlap = overlap[overlap > 1]

if overlap.empty:
    print("\nOK: консервативные группы не пересекаются.")
else:
    print(
        "\nОГРАНИЧЕНИЕ: пересекаются консервативные "
        "группы. Независимость по дате/сессии "
        "не гарантирована."
    )
    print(
        overlap.groupby(level=0)
        .size()
        .to_string()
    )

print("\nOK: каждая отдельная съёмка принадлежит")
print("строго одной из Train/Validation/Test.")

# 6. Фиксируем результат
stage_a_split_df = quality.copy()

stage_a_train_quality = quality[
    quality["split"] == "train"
].copy()

stage_a_val = quality[
    quality["split"] == "val"
].copy()

stage_a_test = quality[
    quality["split"] == "test"
].copy()

# Локальный файл
local_path = Path("/content/stage_a_split_v2.csv")
stage_a_split_df.to_csv(local_path, index=False)

# 7. Сохранение на Google Drive
try:
    from google.colab import drive

    drive.mount("/content/drive")

    save_dir = Path(
        "/content/drive/MyDrive/Magnum_StageA"
    )
    save_dir.mkdir(parents=True, exist_ok=True)

    drive_path = save_dir / "stage_a_split_v2.csv"

    stage_a_split_df.to_csv(
        drive_path, index=False
    )

    distribution.to_csv(
        save_dir / "stage_a_distribution_v2.csv"
    )

    print("\nСохранено в Google Drive:")
    print(drive_path)

except Exception as e:
    print("\nDrive сохранить не удалось:", e)
    print("Локальная копия:", local_path)

print("\nРАЗБИЕНИЕ V2 СФОРМИРОВАНО")


In [ ]:

from google.colab import files
from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 60)
print("MAGNUM — ФИКСАЦИЯ SPLIT V2")
print("=" * 60)

path = Path("/content/stage_a_split_v2.csv")
assert path.exists(), "Файл разбиения не найден!"

saved = pd.read_csv(path)

assert len(saved) == 3723
assert set(saved["split"].unique()) == {
    "train", "val", "test"
}

print("Разбиение сохранено:", path)
print(saved["split"].value_counts().to_string())

print("\nСкачиваем CSV на компьютер...")
files.download(str(path))

print("\n" + "=" * 60)
print("ПРОВЕРКА ДАННЫХ ДЛЯ CNN BALANCED")
print("=" * 60)

balanced = cnn_results["cnn_balanced"]

print("Эмбеддинги E:", E.shape)
print("cnn_balanced:", balanced.shape)
print("Столбцы cnn_balanced:")
print(balanced.columns.tolist())

print("\nПервые 3 строки:")
print(balanced.head(3).to_string(
    index=False, max_colwidth=40
))

if isinstance(cnn, pd.DataFrame):
    print("\nИсходная таблица cnn:")
    print("Размер:", cnn.shape)
    print("Столбцы:", cnn.columns.tolist())

    keys = ["class", "filename"]

    if all(k in cnn.columns and k in balanced.columns
           for k in keys):

        same_order = (
            cnn[keys].reset_index(drop=True).equals(
                balanced[keys].reset_index(drop=True)
            )
        )

        print("Совпадает порядок cnn и cnn_balanced:",
              same_order)

print("\nПредварительная статистика Train:")
train = saved[saved["split"] == "train"].copy()

print("Фильтр качества:", len(train))

if "use_quality_and_cnn" in train.columns:
    mask = (
        train["use_quality_and_cnn"]
        .astype(str).str.lower() == "true"
    )

    print("Старый CNN Balanced внутри Train:",
          int(mask.sum()))

    print("\nПо классам:")
    print(
        pd.crosstab(
            train["class"],
            mask,
            rownames=["class"],
            colnames=["CNN keep"]
        ).to_string()
    )

print("\nГОТОВО")


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import milp, Bounds, LinearConstraint

SEED = 42
TARGET = np.array([0.70, 0.15, 0.15])
SPLITS = ["train", "val", "test"]

# ==========================================
# 1. Исходные данные
# ==========================================

df = split_base_df.copy()

for col in ["use_quality_only", "use_quality_and_cnn"]:
    df[col] = (
        df[col].astype(str)
        .str.strip().str.lower()
        .isin(["true", "1", "yes"])
    )

quality = df.loc[df["use_quality_only"]].copy()

assert len(quality) == 3723
assert df["use_quality_and_cnn"].sum() == 2690
assert quality[["class", "filename", "shooting"]].notna().all().all()
assert not quality.duplicated(["class", "filename"]).any()

quality["split_group"] = (
    quality["class"].astype(str)
    + "||" +
    quality["shooting"].astype(str)
)

quality["split"] = ""

# ==========================================
# 2. Оптимизация по целым съёмкам
# ==========================================

def optimize_groups(sizes, seed, min_train_groups):

    sizes = np.asarray(sizes, dtype=int)
    k = len(sizes)
    total = int(sizes.sum())

    # 3 варианта назначения для каждой съёмки
    # + 3 переменные для оценки отклонения от 70/15/15
    n_vars = 3 * k + 3

    rows, lower, upper = [], [], []

    def add_constraint(terms, lo=-np.inf, hi=np.inf):
        a = np.zeros(n_vars)

        for index, value in terms:
            a[index] = value

        rows.append(a)
        lower.append(lo)
        upper.append(hi)

    # Каждая съёмка строго в одной выборке
    for i in range(k):
        add_constraint(
            [(3 * i + s, 1) for s in range(3)],
            lo=1, hi=1
        )

    # Стремимся к 70/15/15
    for s in range(3):

        terms = [
            (3 * i + s, int(size))
            for i, size in enumerate(sizes)
        ]

        deviation = (3 * k + s, -1)

        add_constraint(
            terms + [deviation],
            hi=total * TARGET[s]
        )

        add_constraint(
            [(idx, -val) for idx, val in terms]
            + [deviation],
            hi=-total * TARGET[s]
        )

    # Минимальное число съёмок
    minimum_groups = [min_train_groups, 2, 2]

    for s in range(3):
        add_constraint(
            [(3 * i + s, 1) for i in range(k)],
            lo=minimum_groups[s]
        )

    # Минимум 10 фото в Validation и Test
    for s in [1, 2]:
        add_constraint(
            [
                (3 * i + s, int(size))
                for i, size in enumerate(sizes)
            ],
            lo=10
        )

    # Не более 85% фотографий класса в Train
    add_constraint(
        [
            (3 * i, int(size))
            for i, size in enumerate(sizes)
        ],
        hi=0.85 * total
    )

    # Минимизируем отклонение от целевых долей
    objective = np.zeros(n_vars)
    objective[3 * k:] = [1, 1, 1]
    objective[3 * k:] /= total

    # Небольшое детерминированное разрешение ничьих
    rng = np.random.default_rng(seed)
    objective[:3 * k] = rng.uniform(
        0, 1e-8, size=3 * k
    )

    variable_max = np.ones(n_vars)
    variable_max[3 * k:] = np.inf

    integrality = np.zeros(n_vars, dtype=int)
    integrality[:3 * k] = 1

    result = milp(
        c=objective,
        integrality=integrality,
        bounds=Bounds(
            np.zeros(n_vars),
            variable_max
        ),
        constraints=LinearConstraint(
            np.asarray(rows),
            np.asarray(lower),
            np.asarray(upper)
        ),
        options={
            "time_limit": 20,
            "mip_rel_gap": 0.02
        }
    )

    if result.x is None:
        raise RuntimeError(
            "Не удалось найти допустимое разбиение: "
            + result.message
        )

    assignments = result.x[:3 * k].reshape(k, 3)

    assert np.all(assignments.max(axis=1) > 0.99)

    return assignments.argmax(axis=1)


# ==========================================
# 3. Разбиение каждого класса
# ==========================================

for index, (cls, part) in enumerate(
    quality.groupby("class", sort=True)
):

    group_sizes = (
        part.groupby("split_group")
        .size()
        .sort_index()
    )

    n_groups = len(group_sizes)

    min_train_groups = 5 if n_groups >= 30 else 3

    assignment = optimize_groups(
        group_sizes.values,
        SEED + index,
        min_train_groups
    )

    group_to_split = dict(zip(
        group_sizes.index,
        [SPLITS[int(x)] for x in assignment]
    ))

    quality.loc[part.index, "split"] = (
        part["split_group"].map(group_to_split)
    )

# ==========================================
# 4. Проверки
# ==========================================

assert quality["split"].isin(SPLITS).all()
assert len(quality) == 3723

# Одна съёмка не может попасть в разные части
assert (
    quality.groupby("split_group")["split"]
    .nunique().max() == 1
)

distribution = pd.crosstab(
    quality["class"], quality["split"]
).reindex(columns=SPLITS, fill_value=0)

group_distribution = (
    quality.groupby(["class", "split"])["split_group"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=SPLITS, fill_value=0)
)

assert (distribution[["val", "test"]] >= 10).all().all()
assert (group_distribution[["val", "test"]] >= 2).all().all()
assert (group_distribution["train"] >= 3).all()

print("=" * 65)
print("MAGNUM — STAGE A / SPLIT V3")
print("=" * 65)

print("\nФотографий по классам:")
print(distribution.to_string())

print("\nСъёмок по классам:")
print(group_distribution.to_string())

print("\nОбщие размеры:")
print(
    quality["split"]
    .value_counts()
    .reindex(SPLITS)
    .to_string()
)

print("\nОбщие доли (%):")
print(
    (
        quality["split"].value_counts(normalize=True)
        .reindex(SPLITS) * 100
    ).round(2).to_string()
)

print("\nМинимум съёмок в Train:",
      group_distribution["train"].min())

print("Пересечений отдельных съёмок: 0")

# Отдельно отмечаем ограничения по датам
overlap = (
    quality.groupby(["class", "conservative_group"])["split"]
    .nunique()
)

overlap = overlap[overlap > 1]

print("\nКонсервативных групп с пересечениями:",
      len(overlap))
print(
    "Оценка независима по отдельным съёмкам, "
    "но не гарантированно по датам."
)

# ==========================================
# 5. Фиксация результата
# ==========================================

stage_a_split_df = quality.copy()

stage_a_train_quality = quality[
    quality["split"] == "train"
].copy()

stage_a_val = quality[
    quality["split"] == "val"
].copy()

stage_a_test = quality[
    quality["split"] == "test"
].copy()

save_path = Path("/content/stage_a_split_v3.csv")

stage_a_split_df.to_csv(
    save_path, index=False, encoding="utf-8"
)

print("\nСохранено:", save_path)

print("\nSTAGE A SPLIT V3 — ГОТОВ К ПРОВЕРКЕ")


In [ ]:

from google.colab import files
from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 65)
print("MAGNUM — СОХРАНЕНИЕ SPLIT V3")
print("=" * 65)

path = Path("/content/stage_a_split_v3.csv")
assert path.exists(), "Не найден файл V3!"

saved = pd.read_csv(path)

assert len(saved) == 3723
assert set(saved["split"]) == {"train", "val", "test"}

print("Размеры:")
print(saved["split"].value_counts().to_string())

print("\nСкачиваем окончательное разбиение...")
files.download(str(path))

print("\n" + "=" * 65)
print("ПОДГОТОВКА CNN BALANCED")
print("=" * 65)

assert isinstance(cnn, pd.DataFrame)
assert isinstance(E, np.ndarray)
assert len(cnn) == len(E), (
    "Количество эмбеддингов не совпадает с cnn"
)

balanced = cnn_results["cnn_balanced"]

print("\nМатрица E:", E.shape)
print("Таблица cnn:", cnn.shape)
print("Таблица cnn_balanced:", balanced.shape)

print("\nСтолбцы cnn:")
print(cnn.columns.tolist())

print("\nСтолбцы cnn_balanced:")
print(balanced.columns.tolist())

keys = ["class", "filename"]

for name, frame in [
    ("cnn", cnn),
    ("cnn_balanced", balanced)
]:
    assert all(k in frame.columns for k in keys), (
        f"{name}: отсутствует class или filename"
    )
    assert not frame.duplicated(keys).any(), (
        f"{name}: повторяются ключи изображений"
    )

same_order = (
    cnn[keys].reset_index(drop=True).equals(
        balanced[keys].reset_index(drop=True)
    )
)

print("\nПорядок фотографий в таблицах совпадает:",
      same_order)

print("\nПервые 5 фотографий cnn:")
print(cnn[keys].head(5).to_string(index=False))

print("\n" + "=" * 65)
print("ПОКРЫТИЕ TRAIN ЭМБЕДДИНГАМИ")
print("=" * 65)

train = saved[saved["split"] == "train"].copy()

matched = train[keys].merge(
    cnn[keys],
    on=keys,
    how="left",
    indicator=True,
    validate="one_to_one"
)

print("Всего Train:", len(train))

print("Есть эмбеддинги:",
      int((matched["_merge"] == "both").sum()))

print("Нет эмбеддингов:",
      int((matched["_merge"] == "left_only").sum()))

print("\nПокрытие по классам:")
print(
    pd.crosstab(
        matched["class"],
        matched["_merge"]
    ).to_string()
)

print("\n" + "=" * 65)
print("СТАРЫЙ CNN BALANCED — ТОЛЬКО ДЛЯ СПРАВКИ")
print("=" * 65)

old_keep = (
    train["use_quality_and_cnn"]
    .astype(str).str.lower().eq("true")
)

print("Сохранено старой фильтрацией:",
      int(old_keep.sum()))

print("\nПо классам:")
print(
    pd.crosstab(
        train["class"],
        old_keep
    ).to_string()
)

print("\nПРОВЕРКА ЗАВЕРШЕНА")


In [ ]:

import pandas as pd
import numpy as np

print("=" * 60)
print("MAGNUM — ПРОВЕРКА CNN ЭМБЕДДИНГОВ")
print("=" * 60)

# Приводим название столбца к единому формату
cnn_norm = cnn.rename(
    columns={"class_name": "class"}
).copy()

balanced_norm = cnn_results["cnn_balanced"].rename(
    columns={"class_name": "class"}
).copy()

keys = ["class", "filename"]

# Проверяем уникальность фотографий
for name, frame in [
    ("cnn", cnn_norm),
    ("cnn_balanced", balanced_norm)
]:
    assert all(k in frame.columns for k in keys)
    assert not frame.duplicated(keys).any(), (
        f"Повторяющиеся фотографии в {name}"
    )

assert len(cnn_norm) == len(E)

# Проверяем совпадение порядка в двух таблицах
same_order = (
    cnn_norm[keys].reset_index(drop=True).equals(
        balanced_norm[keys].reset_index(drop=True)
    )
)

print("Эмбеддинги E:", E.shape)
print("Порядок cnn и cnn_balanced совпадает:", same_order)

# Проверяем покрытие обучающих изображений
train = stage_a_split_df[
    stage_a_split_df["split"] == "train"
].copy()

matched = train[keys].merge(
    cnn_norm[keys],
    on=keys,
    how="left",
    indicator=True,
    validate="one_to_one"
)

print("\nВсего фотографий Train:", len(train))
print("Есть CNN-признаки:",
      (matched["_merge"] == "both").sum())
print("Нет CNN-признаков:",
      (matched["_merge"] == "left_only").sum())

print("\nПокрытие по классам:")
print(
    pd.crosstab(
        matched["class"],
        matched["_merge"]
    ).to_string()
)

print("\nСтатистика предыдущей фильтрации:")
print(
    train["use_quality_and_cnn"]
    .value_counts().to_string()
)

print("\nПРОВЕРКА ЗАВЕРШЕНА")


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 65)
print("MAGNUM — TRAIN-ONLY CNN DIVERSITY")
print("=" * 65)

KEYS = ["class", "filename"]

# =====================================
# 1. Подготовка матрицы и таблицы
# =====================================

meta = cnn.rename(
    columns={"class_name": "class"}
).reset_index(drop=True).copy()

features = np.asarray(E, dtype=np.float32)

assert len(meta) == len(features)
assert not meta.duplicated(KEYS).any()
assert features.shape[1] == 512

norms = np.linalg.norm(
    features, axis=1, keepdims=True
)

assert np.isfinite(features).all()
assert (norms > 0).all()

features = features / norms

meta["embedding_index"] = np.arange(len(meta))

key_to_index = {
    (str(r["class"]), str(r["filename"])): i
    for i, r in meta.iterrows()
}

# =====================================
# 2. Проверяем порядок E через cosine
# =====================================

pairs = []

for i, r in meta.iterrows():

    ref_name = r["matched_filename"]
    stored_cosine = pd.to_numeric(
        r["cosine"], errors="coerce"
    )

    if pd.isna(ref_name) or pd.isna(stored_cosine):
        continue

    j = key_to_index.get(
        (str(r["class"]), str(ref_name))
    )

    if j is None or j == i:
        continue

    pairs.append((i, j, float(stored_cosine)))

print("Найдено проверочных пар:", len(pairs))

if len(pairs) < 10:
    raise RuntimeError(
        "Недостаточно пар для подтверждения порядка E. "
        "Нельзя безопасно использовать эмбеддинги."
    )

# Не более 1000 пар для быстрой проверки
check_ids = np.linspace(
    0, len(pairs) - 1,
    min(1000, len(pairs)),
    dtype=int
)

checked = np.array(
    [pairs[i] for i in check_ids],
    dtype=float
)

left = checked[:, 0].astype(int)
right = checked[:, 1].astype(int)

computed = np.sum(
    features[left] * features[right],
    axis=1
)

errors = np.abs(computed - checked[:, 2])

print("Медианное отличие cosine:",
      round(float(np.median(errors)), 6))

print("Доля совпадений (погрешность до 0.01):",
      round(float((errors < 0.01).mean()), 4))

if (
    np.median(errors) > 0.005
    or (errors < 0.01).mean() < 0.95
):
    raise RuntimeError(
        "Порядок E и cnn не подтверждён. "
        "Отбор остановлен, чтобы не перепутать фотографии."
    )

print("OK: соответствие эмбеддингов подтверждено.")

# =====================================
# 3. Формируем Train с эмбеддингами
# =====================================

train = stage_a_split_df.loc[
    stage_a_split_df["split"] == "train"
].copy()

train = train.merge(
    meta[KEYS + ["embedding_index"]],
    on=KEYS,
    how="left",
    validate="one_to_one"
)

with_features = train[
    train["embedding_index"].notna()
].copy()

without_features = train[
    train["embedding_index"].isna()
].copy()

assert len(train) == 2488

# Фиксируем долю отбора заранее
KEEP_RATIO = 2690 / 3723

target_total = round(len(train) * KEEP_RATIO)
target_embedded = target_total - len(without_features)

print("\nОбычный Train:", len(train))
print("Без эмбеддингов, оставляем:",
      len(without_features))
print("Целевой размер CNN Train:", target_total)

# =====================================
# 4. Распределяем квоты по съёмкам
# =====================================

group_cols = ["class", "shooting"]

group_sizes = (
    with_features.groupby(group_cols)
    .size()
    .rename("n")
    .to_frame()
)

ratio_embedded = (
    target_embedded / len(with_features)
)

group_sizes["ideal"] = (
    group_sizes["n"] * ratio_embedded
)

group_sizes["take"] = np.maximum(
    1,
    np.floor(group_sizes["ideal"]).astype(int)
)

# Доводим сумму квот до целевого размера
while group_sizes["take"].sum() < target_embedded:

    eligible = group_sizes[
        group_sizes["take"] < group_sizes["n"]
    ]

    assert not eligible.empty

    idx = (
        eligible["ideal"] - eligible["take"]
    ).idxmax()

    group_sizes.loc[idx, "take"] += 1

while group_sizes["take"].sum() > target_embedded:

    eligible = group_sizes[
        group_sizes["take"] > 1
    ]

    assert not eligible.empty

    idx = (
        eligible["take"] - eligible["ideal"]
    ).idxmax()

    group_sizes.loc[idx, "take"] -= 1

# =====================================
# 5. Выбираем разнообразные изображения
# =====================================

def select_diverse_positions(block, amount):

    if amount >= len(block):
        return block.index.tolist()

    embeddings_idx = (
        block["embedding_index"].astype(int).to_numpy()
    )

    vectors = features[embeddings_idx]

    # Чем выше cosine, тем похожее фотографии
    similarities = vectors @ vectors.T

    # Начинаем с наиболее типичного изображения
    first = int(np.argmax(
        similarities.mean(axis=1)
    ))

    selected = [first]

    # Для каждой фотографии ищем сходство
    # с ближайшим уже выбранным изображением
    max_similarity = similarities[first].copy()
    max_similarity[first] = np.inf

    while len(selected) < amount:

        # Выбираем наименее похожую
        # на уже сохранённые фотографии
        next_pos = int(np.argmin(max_similarity))

        selected.append(next_pos)

        max_similarity = np.maximum(
            max_similarity,
            similarities[next_pos]
        )

        max_similarity[selected] = np.inf

    return block.index[selected].tolist()


selected_positions = without_features.index.tolist()

for group_key, block in with_features.groupby(
    group_cols, sort=True
):

    amount = int(
        group_sizes.loc[group_key, "take"]
    )

    selected_positions.extend(
        select_diverse_positions(block, amount)
    )

# =====================================
# 6. Готовый второй Train
# =====================================

stage_a_train_cnn_balanced = (
    train.loc[sorted(selected_positions)]
    .drop(columns=["embedding_index"])
    .copy()
)

stage_a_train_quality = (
    train.drop(columns=["embedding_index"])
    .copy()
)

stage_a_val = stage_a_split_df[
    stage_a_split_df["split"] == "val"
].copy()

stage_a_test = stage_a_split_df[
    stage_a_split_df["split"] == "test"
].copy()

assert len(stage_a_train_cnn_balanced) == target_total

keys_quality = set(
    map(tuple, stage_a_train_quality[KEYS].to_numpy())
)

keys_balanced = set(
    map(tuple, stage_a_train_cnn_balanced[KEYS].to_numpy())
)

assert keys_balanced.issubset(keys_quality)
assert len(keys_balanced) == len(stage_a_train_cnn_balanced)

# Нельзя пересекаться с Validation и Test
held_out_keys = set(map(
    tuple,
    pd.concat([stage_a_val, stage_a_test])[KEYS]
      .to_numpy()
))

assert keys_balanced.isdisjoint(held_out_keys)
assert keys_quality.isdisjoint(held_out_keys)

# =====================================
# 7. Отчёт и сохранение
# =====================================

print("\n" + "=" * 65)
print("РЕЗУЛЬТАТ")
print("=" * 65)

print("Train 1 (Quality):",
      len(stage_a_train_quality))

print("Train 2 (CNN Diversity):",
      len(stage_a_train_cnn_balanced))

print("Validation (общая):", len(stage_a_val))
print("Test (общий):", len(stage_a_test))

print("\nСравнение по классам:")

comparison = pd.DataFrame({
    "Quality": stage_a_train_quality[
        "class"
    ].value_counts(),
    "CNN_Diversity": stage_a_train_cnn_balanced[
        "class"
    ].value_counts()
}).fillna(0).astype(int)

print(comparison.to_string())

print("\nСъёмок во втором Train:")

print(
    stage_a_train_cnn_balanced.groupby("class")[
        "shooting"
    ].nunique().to_string()
)

output = Path(
    "/content/stage_a_train_cnn_balanced_v3.csv"
)

stage_a_train_cnn_balanced.to_csv(
    output, index=False
)

print("\nСохранено:", output)
print("ГОТОВО: ОБА TRAIN ПОДГОТОВЛЕНЫ")


In [ ]:
from google.colab import files

files.download(
    "/content/stage_a_train_cnn_balanced_v3.csv"
)